# Load DRIAMS and MARISMA datasets

In this step we load the two datasets used in the study:

- **DRIAMS**: large clinical MALDI-TOF dataset
- **MARISMA**: curated dataset from the MARISMA study

Each pickle file contains a dictionary with the following keys:

- `data`: MALDI spectra matrix
- `label`: bacterial species
- `amr`: antimicrobial resistance matrix
- `antibiotics`: list of antibiotics corresponding to the AMR matrix columns

We load both datasets and inspect their structure to verify that the format is compatible.

In [46]:
import pickle
import numpy as np
import pandas as pd

driams_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/DRIAMS_A_AMR_whole_pipeline_23_DRIAMS.pkl"
marisma_path = "/export/usuarios01/egarroyo/MALDI_for_AMR_prediction/data/MARISMa_study_MARISMA_whole_pipeline.pkl"

with open(driams_path, "rb") as f:
    driams_payload = pickle.load(f)

with open(marisma_path, "rb") as f:
    marisma_payload = pickle.load(f)

print("Datasets loaded.")


def inspect_dataset(payload, name):
    X = payload["data"]
    y_species = payload["label"]
    amr = payload["amr"]
    antibiotics = payload["antibiotics"]

    print(f"\n{name}")
    print("-" * 50)
    print("Spectra shape:", X.shape)
    print("AMR matrix shape:", amr.shape)
    print("Number of antibiotics:", len(antibiotics))
    print("Unique species:", np.unique(y_species))


inspect_dataset(driams_payload, "DRIAMS")
inspect_dataset(marisma_payload, "MARISMA")

Datasets loaded.

DRIAMS
--------------------------------------------------
Spectra shape: (18168, 6000)
AMR matrix shape: (18168, 9)
Number of antibiotics: 9
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']

MARISMA
--------------------------------------------------
Spectra shape: (21360, 6000)
AMR matrix shape: (21360, 8)
Number of antibiotics: 8
Unique species: ['Escherichia_Coli' 'Klebsiella_Pneumoniae' 'Pseudomonas_Aeruginosa'
 'Staphylococcus_Aureus']


# Combine DRIAMS and MARISMA datasets

To increase statistical power we merge both datasets.

The merge is possible because both datasets share the same structure:
- spectra matrix
- species labels
- AMR matrix
- antibiotic list

After merging we obtain a single dataset containing all spectra and AMR annotations.

In [47]:
species_antibiotics = {
    "Staphylococcus_Aureus": [
        "Oxacillin", "Clindamycin", "Fusidic acid"
    ],
    "Escherichia_Coli": [
        "Ciprofloxacin", "Ceftriaxone", "Cefepime"
    ],
    "Klebsiella_Pneumoniae": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ],
    "Pseudomonas_Aeruginosa": [
        "Ciprofloxacin", "Imipenem", "Meropenem"
    ]
}

In [48]:
# Union of all antibiotics used in the analysis

selected_antibiotics = sorted(
    {ab for abs_list in species_antibiotics.values() for ab in abs_list}
)

print("Selected antibiotics:")
print(selected_antibiotics)

Selected antibiotics:
['Cefepime', 'Ceftriaxone', 'Ciprofloxacin', 'Clindamycin', 'Fusidic acid', 'Imipenem', 'Meropenem', 'Oxacillin']


In [8]:
def extract_selected_amr(payload, selected_antibiotics):

    antibiotics_dataset = payload["antibiotics"]
    amr_dataset = payload["amr"]

    antibiotic_index = {a: i for i, a in enumerate(antibiotics_dataset)}

    amr_selected = np.zeros((amr_dataset.shape[0], len(selected_antibiotics)))

    for j, ab in enumerate(selected_antibiotics):

        if ab in antibiotic_index:
            amr_selected[:, j] = amr_dataset[:, antibiotic_index[ab]]

    return amr_selected

In [49]:
amr_driams = extract_selected_amr(driams_payload, selected_antibiotics)
amr_marisma = extract_selected_amr(marisma_payload, selected_antibiotics)

print("DRIAMS AMR shape:", amr_driams.shape)
print("MARISMA AMR shape:", amr_marisma.shape)

DRIAMS AMR shape: (18168, 8)
MARISMA AMR shape: (21360, 8)


In [50]:
X_all = np.concatenate(
    [driams_payload["data"], marisma_payload["data"]],
    axis=0
)

y_all = np.concatenate(
    [driams_payload["label"], marisma_payload["label"]],
    axis=0
)

amr_all = np.concatenate(
    [amr_driams, amr_marisma],
    axis=0
)

print("Combined dataset shapes")
print("Spectra:", X_all.shape)
print("Species:", y_all.shape)
print("AMR:", amr_all.shape)

Combined dataset shapes
Spectra: (39528, 6000)
Species: (39528,)
AMR: (39528, 8)


# Clinical resistance distribution per species

In this analysis we classify isolates into three clinically meaningful groups
based on the number of resistant antibiotics:

- **Susceptible (S)** → resistant to 0 antibiotics
- **Single Resistance (SR)** → resistant to exactly 1 antibiotic
- **Multiple Resistance (MR)** → resistant to more than 1 antibiotic

This classification is biologically meaningful because isolates with multiple
resistance mechanisms often exhibit stronger phenotypic changes in MALDI-TOF
spectra (e.g. membrane permeability changes or efflux systems), while single
resistance events may correspond to more subtle molecular changes.

For each species we compute the number of samples belonging to each category.

In [51]:
import pandas as pd
import numpy as np

# mapping antibiotic → column index
antibiotic_index = {a: i for i, a in enumerate(selected_antibiotics)}

summary_rows = []

for species, ab_list in species_antibiotics.items():

    # select antibiotics relevant for that species
    idx = [antibiotic_index[a] for a in ab_list]

    # select species samples
    mask = y_all == species
    amr_species = amr_all[mask][:, idx]

    n_samples = amr_species.shape[0]

    # count number of resistances per isolate
    resistance_counts = amr_species.sum(axis=1)

    susceptible = np.sum(resistance_counts == 0)
    single = np.sum(resistance_counts == 1)
    multi = np.sum(resistance_counts > 1)

    summary_rows.append({
        "species": species,
        "total_samples": n_samples,
        "susceptible": susceptible,
        "single_resistance": single,
        "multiple_resistance": multi
    })

summary_df = pd.DataFrame(summary_rows)

print(summary_df)

                  species  total_samples  susceptible  single_resistance  \
0   Staphylococcus_Aureus           7224         3260               1001   
1        Escherichia_Coli           7716         3723                784   
2   Klebsiella_Pneumoniae          19259        13592               4230   
3  Pseudomonas_Aeruginosa           5329         1918                616   

   multiple_resistance  
0                  388  
1                 1116  
2                  570  
3                 1314  


# Exact resistance pattern (LPS) distribution

Here we compute the frequency of each **exact resistance pattern (LPS)**.

An LPS pattern corresponds to the binary resistance vector across the selected
antibiotics for a given species.

Example for 3 antibiotics:

(0,0,0) → fully susceptible  
(1,0,0) → resistance to antibiotic 1 only  
(1,1,0) → resistance to antibiotics 1 and 2  
(1,1,1) → resistance to all three  

Counting these patterns allows us to understand:

- class imbalance
- prevalence of single vs multiple resistance combinations
- which patterns dominate the dataset

In [52]:
from collections import Counter

for species, ab_list in species_antibiotics.items():

    print("\n" + "="*60)
    print(species)
    print("="*60)

    idx = [antibiotic_index[a] for a in ab_list]

    mask = y_all == species
    amr_species = amr_all[mask][:, idx]

    # Convert binary AMR rows to S/R patterns
    patterns = []
    for row in amr_species:
        sr_pattern = tuple("R" if x == 1 else "S" for x in row)
        patterns.append(sr_pattern)

    pattern_counts = Counter(patterns)

    print("Antibiotics:", ab_list)
    print("\nPattern counts:\n")

    for pattern, count in sorted(pattern_counts.items(), key=lambda x: -x[1]):

        pattern_str = " ".join(pattern)

        print(f"{pattern_str} : {count}")


Staphylococcus_Aureus
Antibiotics: ['Oxacillin', 'Clindamycin', 'Fusidic acid']

Pattern counts:

S S S : 4959
R S S : 835
S R S : 767
R R S : 273
S S R : 204
R S R : 108
S R R : 48
R R R : 30

Escherichia_Coli
Antibiotics: ['Ciprofloxacin', 'Ceftriaxone', 'Cefepime']

Pattern counts:

S S S : 4951
R S S : 1141
R R R : 722
R S R : 286
S R R : 252
R R S : 202
S R S : 106
S S R : 56

Klebsiella_Pneumoniae
Antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']

Pattern counts:

S S S : 14214
R S S : 4375
R R R : 359
R R S : 152
R S R : 58
S R S : 42
S R R : 39
S S R : 20

Pseudomonas_Aeruginosa
Antibiotics: ['Ciprofloxacin', 'Imipenem', 'Meropenem']

Pattern counts:

S S S : 2762
R R S : 862
S R S : 511
R S S : 426
R R R : 331
S R R : 197
R S R : 152
S S R : 88


# Balanced training subset construction

The dataset is highly imbalanced because susceptible isolates are much more
frequent than resistant ones.

To avoid biasing the classifier towards the susceptible class we construct
a balanced training subset for each species using the following strategy:

- keep **all single resistance isolates**
- keep **all multiple resistance isolates**
- randomly sample only a fraction of **susceptible isolates**

The number of susceptible isolates is defined as:

n_susceptible = α × n_resistant

where

n_resistant = n_single + n_multiple

This produces a dataset that preserves all resistance patterns while reducing
the dominance of susceptible samples.

In [53]:
import numpy as np

alpha = 1.5
rng = np.random.default_rng(42)

selected_indices = []

for species, ab_list in species_antibiotics.items():

    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask_species = (y_all == species)
    species_indices = np.where(mask_species)[0]

    amr_species = amr_all[species_indices][:, idx_ab]

    resistance_counts = amr_species.sum(axis=1)

    susceptible_idx = species_indices[resistance_counts == 0]
    single_idx = species_indices[resistance_counts == 1]
    multi_idx = species_indices[resistance_counts > 1]

    n_resistant = len(single_idx) + len(multi_idx)
    n_susceptible_target = int(alpha * n_resistant)

    if len(susceptible_idx) > n_susceptible_target:
        susceptible_sample = rng.choice(
            susceptible_idx,
            size=n_susceptible_target,
            replace=False
        )
    else:
        susceptible_sample = susceptible_idx

    selected_indices.extend(single_idx)
    selected_indices.extend(multi_idx)
    selected_indices.extend(susceptible_sample)

    print("\n", species)
    print("single:", len(single_idx))
    print("multi:", len(multi_idx))
    print("susceptible used:", len(susceptible_sample))

selected_indices = np.array(selected_indices)


 Staphylococcus_Aureus
single: 1001
multi: 388
susceptible used: 2083

 Escherichia_Coli
single: 784
multi: 1116
susceptible used: 2850

 Klebsiella_Pneumoniae
single: 4230
multi: 570
susceptible used: 7200

 Pseudomonas_Aeruginosa
single: 616
multi: 1314
susceptible used: 1918


In [16]:
X_balanced = X_all[selected_indices]
y_balanced = y_all[selected_indices]
amr_balanced = amr_all[selected_indices]

print("Balanced dataset shape:", X_balanced.shape)

Balanced dataset shape: (24070, 6000)


# Binary classification task: multi-resistant vs non-multi-resistant

We formulate the first experiment as a **binary classification problem**.

The goal of the model is to detect **multi-resistant isolates**, which are
biologically expected to exhibit stronger phenotypic shifts in MALDI-TOF spectra.

Classes are defined as:

0 → susceptible + single resistance  
1 → multiple resistance  

This classifier acts as the **first stage of a hierarchical AMR prediction system**.

In [17]:
y_binary = np.zeros(len(selected_indices))

for i, idx in enumerate(selected_indices):

    species = y_all[idx]
    ab_list = species_antibiotics[species]
    idx_ab = [antibiotic_index[a] for a in ab_list]

    resistance_count = amr_all[idx, idx_ab].sum()

    if resistance_count > 1:
        y_binary[i] = 1

y_binary = y_binary.astype(int)

print("Label distribution")
print("Non-multi:", np.sum(y_binary == 0))
print("Multi:", np.sum(y_binary == 1))

Label distribution
Non-multi: 20682
Multi: 3388


In [18]:
X_balanced = X_all[selected_indices]

print("Dataset shape:", X_balanced.shape)

Dataset shape: (24070, 6000)


# Train / validation / test split

To properly evaluate the model we split the dataset into:

- **Training set** (70%)
- **Validation set** (15%)
- **Test set** (15%)

The split is stratified to preserve the proportion of multi-resistant isolates.

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X_balanced,
    y_binary,
    test_size=0.30,
    stratify=y_binary,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (16849, 6000)
Validation: (3610, 6000)
Test: (3611, 6000)


In [20]:
import torch
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

batch_size = 128

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=batch_size
)

test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=batch_size
)

 # Species-specific multi-resistance classifier

In this experiment we train **one neural network per bacterial species**
to detect **multi-resistant isolates** using MALDI-TOF spectra.

The classification task is formulated as:

0 → susceptible + single resistance  
1 → multiple resistance  

This classifier represents the **first stage of a hierarchical AMR prediction
pipeline**, where the goal is to identify isolates with strong phenotypic
changes associated with multiple resistance mechanisms.

We train a **species-specific MLP model** because MALDI-TOF spectra are strongly
species-dependent. Training a single model across species would introduce
biological confounding, as spectral differences between species are much
larger than resistance-associated signals.

The neural network follows the **MLP architecture (size M)** described in the
reference paper:

6000 → 512 → 256 → 128 → 64 → 1

Between layers we apply:

• GeLU activation  
• Layer normalization  
• Dropout (0.2)

Training uses:

• AdamW optimizer  
• BCEWithLogitsLoss  
• Early stopping (patience = 25) based on validation AUC

In [31]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"

alpha = 1.5
rng = np.random.default_rng(42)

antibiotic_index = {a: i for i, a in enumerate(selected_antibiotics)}

In [32]:
class MLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(6000,512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),

            nn.Linear(512,256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),

            nn.Linear(256,128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128,64),
            nn.GELU(),
            nn.LayerNorm(64),
            nn.Dropout(0.2),

            nn.Linear(64,1)
        )

    def forward(self,x):

        return self.net(x).squeeze()

# Training loop per species

For each species we:

1. Extract spectra of that species
2. Compute binary labels (multi vs non-multi)
3. Construct a balanced dataset
4. Perform train / validation / test split
5. Train the MLP with early stopping
6. Evaluate on the test set

In [35]:
class FocalLoss(nn.Module):

    def __init__(self, alpha=0.75, gamma=2):

        super().__init__()

        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):

        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits)

        pt = torch.where(targets == 1, probs, 1 - probs)

        focal_weight = self.alpha * (1 - pt) ** self.gamma

        loss = focal_weight * bce_loss

        return loss.mean()

In [37]:
results = {}

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*60)
    print("TRAINING:", species)
    print("="*60)

    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask_species = y_all == species
    species_indices = np.where(mask_species)[0]

    amr_species = amr_all[species_indices][:, idx_ab]

    resistance_counts = amr_species.sum(axis=1)

    susceptible_idx = species_indices[resistance_counts == 0]
    single_idx = species_indices[resistance_counts == 1]
    multi_idx = species_indices[resistance_counts > 1]

    n_resistant = len(single_idx) + len(multi_idx)
    n_sus_target = int(alpha * n_resistant)

    if len(susceptible_idx) > n_sus_target:
        susceptible_sample = rng.choice(susceptible_idx,n_sus_target,replace=False)
    else:
        susceptible_sample = susceptible_idx

    selected = np.concatenate([
        single_idx,
        multi_idx,
        susceptible_sample
    ])

    X = X_all[selected]

    y = []

    for idx in selected:

        count = amr_all[idx,idx_ab].sum()

        if count > 1:
            y.append(1)
        else:
            y.append(0)

    y = np.array(y)

    print("Samples:",len(X))
    print("Multi:",np.sum(y==1))
    print("Non-multi:",np.sum(y==0))

    X_train,X_temp,y_train,y_temp = train_test_split(
        X,y,test_size=0.3,stratify=y,random_state=42
    )

    X_val,X_test,y_val,y_test = train_test_split(
        X_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42
    )

    X_train = torch.tensor(X_train,dtype=torch.float32)
    X_val = torch.tensor(X_val,dtype=torch.float32)
    X_test = torch.tensor(X_test,dtype=torch.float32)

    y_train = torch.tensor(y_train,dtype=torch.float32)
    y_val = torch.tensor(y_val,dtype=torch.float32)
    y_test = torch.tensor(y_test,dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(X_train,y_train),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val,y_val),
        batch_size=128
    )

    test_loader = DataLoader(
        TensorDataset(X_test,y_test),
        batch_size=128
    )

    model = MLP().to(device)

    n_pos = np.sum(y_train.numpy()==1)
    n_neg = np.sum(y_train.numpy()==0)

    pos_weight = torch.tensor([n_neg/n_pos]).to(device)

    criterion = FocalLoss(alpha=0.5, gamma=1.5)

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

    best_auc = 0
    patience = 5
    epochs_no_improve = 0
    best_model = None

    for epoch in range(200):

        model.train()

        for Xb,yb in train_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(Xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

        model.eval()

        preds = []
        targets = []

        with torch.no_grad():

            for Xb,yb in val_loader:

                Xb = Xb.to(device)

                logits = model(Xb)

                probs = torch.sigmoid(logits).cpu().numpy()

                preds.extend(probs)
                targets.extend(yb.numpy())

        auc = roc_auc_score(targets,preds)

        print(f"Epoch {epoch:03d}  Val AUC {auc:.4f}")

        if auc > best_auc:

            best_auc = auc
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            print("Early stopping")
            break

    model.load_state_dict(best_model)

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for Xb,yb in test_loader:

            Xb = Xb.to(device)

            logits = model(Xb)

            probs = torch.sigmoid(logits).cpu().numpy()

            preds.extend(probs)
            targets.extend(yb.numpy())

    preds_bin = (np.array(preds)>0.5).astype(int)

    test_auc = roc_auc_score(targets,preds)
    acc = accuracy_score(targets,preds_bin)
    cm = confusion_matrix(targets,preds_bin)

    print("\nTest AUC:",test_auc)
    print("Accuracy:",acc)
    print("Confusion matrix:\n",cm)

    results[species] = test_auc


TRAINING: Staphylococcus_Aureus
Samples: 3472
Multi: 388
Non-multi: 3084
Epoch 000  Val AUC 0.7704
Epoch 001  Val AUC 0.8021
Epoch 002  Val AUC 0.8207
Epoch 003  Val AUC 0.8407
Epoch 004  Val AUC 0.8374
Epoch 005  Val AUC 0.8388
Epoch 006  Val AUC 0.8440
Epoch 007  Val AUC 0.8366
Epoch 008  Val AUC 0.8329
Epoch 009  Val AUC 0.8226
Epoch 010  Val AUC 0.8254
Epoch 011  Val AUC 0.8267
Early stopping

Test AUC: 0.8648246071348775
Accuracy: 0.8771593090211133
Confusion matrix:
 [[416  47]
 [ 17  41]]

TRAINING: Escherichia_Coli
Samples: 4750
Multi: 1116
Non-multi: 3634
Epoch 000  Val AUC 0.6813
Epoch 001  Val AUC 0.7082
Epoch 002  Val AUC 0.7887
Epoch 003  Val AUC 0.8164
Epoch 004  Val AUC 0.8155
Epoch 005  Val AUC 0.8072
Epoch 006  Val AUC 0.8380
Epoch 007  Val AUC 0.8309
Epoch 008  Val AUC 0.8156
Epoch 009  Val AUC 0.8411
Epoch 010  Val AUC 0.8417
Epoch 011  Val AUC 0.8361
Epoch 012  Val AUC 0.8090
Epoch 013  Val AUC 0.8233
Epoch 014  Val AUC 0.8203
Epoch 015  Val AUC 0.8304
Early stoppi

# Multiclass classification of multi-resistance patterns

In the second stage of the hierarchical pipeline we train a classifier
to distinguish between **different multi-resistance patterns**.

This model is trained **only on isolates exhibiting multiple resistance**.

The task becomes a standard multiclass classification problem where
each class corresponds to a specific resistance pattern across the
antibiotics considered for a given species.

For example with three antibiotics:

R R S  
R S R  
S R R  
R R R

The same MLP architecture used in the binary classifier is employed,
but the output layer now predicts the number of multi-resistance patterns.

In [63]:
from collections import Counter
import copy

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*60)
    print("MULTICLASS TRAINING:", species)
    print("="*60)

    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask_species = y_all == species
    species_indices = np.where(mask_species)[0]

    amr_species = amr_all[species_indices][:, idx_ab]

    resistance_counts = amr_species.sum(axis=1)

    multi_mask = resistance_counts > 1
    multi_indices = species_indices[multi_mask]

    X_multi = X_all[multi_indices]

    patterns = []

    for idx in multi_indices:
        pattern = tuple(amr_all[idx, idx_ab].astype(int))
        patterns.append(pattern)

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    y_multi = np.array([pattern_to_class[p] for p in patterns])

    print("Multi samples:",len(X_multi))
    print("Classes:",len(unique_patterns))

    if len(unique_patterns) < 2:
        print("Skipping species (not enough classes)")
        continue

    # -------------------------
    # train / val / test split
    # -------------------------

    X_train,X_temp,y_train,y_temp = train_test_split(
        X_multi,y_multi,test_size=0.3,stratify=y_multi,random_state=42
    )

    X_val,X_test,y_val,y_test = train_test_split(
        X_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42
    )

    X_train = torch.tensor(X_train,dtype=torch.float32)
    X_val = torch.tensor(X_val,dtype=torch.float32)
    X_test = torch.tensor(X_test,dtype=torch.float32)

    y_train = torch.tensor(y_train,dtype=torch.long)
    y_val = torch.tensor(y_val,dtype=torch.long)
    y_test = torch.tensor(y_test,dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train,y_train),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val,y_val),
        batch_size=128
    )

    test_loader = DataLoader(
        TensorDataset(X_test,y_test),
        batch_size=128
    )

    # -------------------------
    # Model
    # -------------------------

    class MultiMLP(nn.Module):

        def __init__(self,n_classes):

            super().__init__()

            self.net = nn.Sequential(

                nn.Linear(6000,512),
                nn.GELU(),
                nn.LayerNorm(512),
                nn.Dropout(0.2),

                nn.Linear(512,256),
                nn.GELU(),
                nn.LayerNorm(256),
                nn.Dropout(0.2),

                nn.Linear(256,128),
                nn.GELU(),
                nn.LayerNorm(128),
                nn.Dropout(0.2),

                nn.Linear(128,64),
                nn.GELU(),
                nn.LayerNorm(64),
                nn.Dropout(0.2),

                nn.Linear(64,n_classes)
            )

        def forward(self,x):
            return self.net(x)

    model = MultiMLP(len(unique_patterns)).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

    # -------------------------
    # Early stopping params
    # -------------------------

    best_auc = -np.inf
    patience = 5
    epochs_no_improve = 0
    best_model = None

    # -------------------------
    # Training loop
    # -------------------------

    for epoch in range(200):

        model.train()

        for Xb,yb in train_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(Xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

        # -------------------------
        # Validation
        # -------------------------

        model.eval()

        preds = []
        targets = []

        with torch.no_grad():

            for Xb,yb in val_loader:

                Xb = Xb.to(device)
                yb = yb.to(device)

                logits = model(Xb)

                probs = torch.softmax(logits,dim=1).cpu().numpy()

                preds.extend(probs)
                targets.extend(yb.cpu().numpy())

        preds = np.array(preds)

        auc = roc_auc_score(targets,preds,multi_class="ovr")

        print(f"Epoch {epoch:03d}  Val AUC {auc:.4f}")

        if auc > best_auc:

            best_auc = auc
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            print("Early stopping triggered")
            break

    model.load_state_dict(best_model)

    print("Best Val AUC:",best_auc)


MULTICLASS TRAINING: Staphylococcus_Aureus
Multi samples: 388
Classes: 4
Epoch 000  Val AUC 0.6236
Epoch 001  Val AUC 0.6168
Epoch 002  Val AUC 0.6284
Epoch 003  Val AUC 0.6415
Epoch 004  Val AUC 0.6624
Epoch 005  Val AUC 0.6794
Epoch 006  Val AUC 0.6942
Epoch 007  Val AUC 0.6920
Epoch 008  Val AUC 0.7165
Epoch 009  Val AUC 0.7001
Epoch 010  Val AUC 0.6950
Epoch 011  Val AUC 0.6987
Epoch 012  Val AUC 0.7283
Epoch 013  Val AUC 0.7394
Epoch 014  Val AUC 0.7394
Epoch 015  Val AUC 0.7628
Epoch 016  Val AUC 0.7384
Epoch 017  Val AUC 0.7537
Epoch 018  Val AUC 0.7702
Epoch 019  Val AUC 0.7783
Epoch 020  Val AUC 0.7777
Epoch 021  Val AUC 0.7586
Epoch 022  Val AUC 0.7569
Epoch 023  Val AUC 0.7226
Epoch 024  Val AUC 0.7154
Early stopping triggered
Best Val AUC: 0.7783482029762809

MULTICLASS TRAINING: Escherichia_Coli
Multi samples: 1116
Classes: 4
Epoch 000  Val AUC 0.6708
Epoch 001  Val AUC 0.6731
Epoch 002  Val AUC 0.6861
Epoch 003  Val AUC 0.6875
Epoch 004  Val AUC 0.6770
Epoch 005  Val AUC

In [64]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

model.eval()

preds = []
targets = []

with torch.no_grad():

    for Xb, yb in test_loader:

        Xb = Xb.to(device)

        logits = model(Xb)

        probs = torch.softmax(logits, dim=1)

        pred_class = torch.argmax(probs, dim=1).cpu().numpy()

        preds.extend(pred_class)
        targets.extend(yb.numpy())

preds = np.array(preds)
targets = np.array(targets)

print("\nTest Accuracy:", accuracy_score(targets, preds))
print("Macro F1:", f1_score(targets, preds, average="macro"))

print("\nConfusion Matrix")
print(confusion_matrix(targets, preds))

print("\nClassification Report")
print(classification_report(targets, preds))


Test Accuracy: 0.7525252525252525
Macro F1: 0.6701361246252887

Confusion Matrix
[[ 15   5   5]
 [  1 113   9]
 [  4  25  21]]

Classification Report
              precision    recall  f1-score   support

           0       0.75      0.60      0.67        25
           1       0.79      0.92      0.85       123
           2       0.60      0.42      0.49        50

    accuracy                           0.75       198
   macro avg       0.71      0.65      0.67       198
weighted avg       0.74      0.75      0.74       198



In [65]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*70)
    print("MULTICLASS TRAINING:", species)
    print("="*70)

    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask_species = y_all == species
    species_indices = np.where(mask_species)[0]

    amr_species = amr_all[species_indices][:, idx_ab]

    resistance_counts = amr_species.sum(axis=1)

    multi_mask = resistance_counts > 1

    multi_indices = species_indices[multi_mask]

    X_multi = X_all[multi_indices]

    patterns = []

    for idx in multi_indices:

        pattern = tuple(amr_all[idx, idx_ab].astype(int))
        patterns.append(pattern)

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    y_multi = np.array([pattern_to_class[p] for p in patterns])

    print("Multi samples:",len(X_multi))
    print("Number of classes:",len(unique_patterns))

    # train / val / test split
    X_train,X_temp,y_train,y_temp = train_test_split(
        X_multi,y_multi,test_size=0.30,stratify=y_multi,random_state=42
    )

    X_val,X_test,y_val,y_test = train_test_split(
        X_temp,y_temp,test_size=0.50,stratify=y_temp,random_state=42
    )

    X_train = torch.tensor(X_train,dtype=torch.float32)
    X_val = torch.tensor(X_val,dtype=torch.float32)
    X_test = torch.tensor(X_test,dtype=torch.float32)

    y_train = torch.tensor(y_train,dtype=torch.long)
    y_val = torch.tensor(y_val,dtype=torch.long)
    y_test = torch.tensor(y_test,dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train,y_train),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val,y_val),
        batch_size=128
    )

    test_loader = DataLoader(
        TensorDataset(X_test,y_test),
        batch_size=128
    )

    # -------- MODEL --------

    class MultiMLP(nn.Module):

        def __init__(self,n_classes):

            super().__init__()

            self.net = nn.Sequential(

                nn.Linear(6000,512),
                nn.GELU(),
                nn.LayerNorm(512),
                nn.Dropout(0.2),

                nn.Linear(512,256),
                nn.GELU(),
                nn.LayerNorm(256),
                nn.Dropout(0.2),

                nn.Linear(256,128),
                nn.GELU(),
                nn.LayerNorm(128),
                nn.Dropout(0.2),

                nn.Linear(128,64),
                nn.GELU(),
                nn.LayerNorm(64),
                nn.Dropout(0.2),

                nn.Linear(64,n_classes)
            )

        def forward(self,x):

            return self.net(x)

    model = MultiMLP(len(unique_patterns)).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

    best_auc = -np.inf
    patience = 25
    epochs_no_improve = 0
    best_model = None

    # -------- TRAINING --------

    for epoch in range(200):

        model.train()

        for Xb,yb in train_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(Xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

        # -------- VALIDATION --------

        model.eval()

        preds = []
        targets = []

        with torch.no_grad():

            for Xb,yb in val_loader:

                Xb = Xb.to(device)

                logits = model(Xb)

                probs = torch.softmax(logits,dim=1).cpu().numpy()

                preds.extend(probs)
                targets.extend(yb.numpy())

        preds = np.array(preds)

        auc = roc_auc_score(targets,preds,multi_class="ovr")

        print(f"Epoch {epoch:03d}  Val AUC {auc:.4f}")

        if auc > best_auc:

            best_auc = auc
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            print("Early stopping triggered")
            break

    model.load_state_dict(best_model)

    print("\nBest Validation AUC:",best_auc)

    # -------- TEST EVALUATION --------

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for Xb,yb in test_loader:

            Xb = Xb.to(device)

            logits = model(Xb)

            probs = torch.softmax(logits,dim=1)

            pred_class = torch.argmax(probs,dim=1).cpu().numpy()

            preds.extend(pred_class)
            targets.extend(yb.numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    print("\nTest Accuracy:",accuracy_score(targets,preds))
    print("Macro F1:",f1_score(targets,preds,average="macro"))

    print("\nConfusion Matrix")
    print(confusion_matrix(targets,preds))

    print("\nClassification Report")
    print(classification_report(targets,preds))

    # -------- CLASS MAPPING --------

    print("\nClass ↔ Resistance pattern")

    for pattern,cls in pattern_to_class.items():

        sr_pattern = ["R" if x==1 else "S" for x in pattern]

        print(f"class {cls} → {' '.join(sr_pattern)}")


MULTICLASS TRAINING: Staphylococcus_Aureus
Multi samples: 388
Number of classes: 4
Epoch 000  Val AUC 0.5815
Epoch 001  Val AUC 0.5952
Epoch 002  Val AUC 0.6199
Epoch 003  Val AUC 0.6419
Epoch 004  Val AUC 0.6483
Epoch 005  Val AUC 0.6527
Epoch 006  Val AUC 0.6639
Epoch 007  Val AUC 0.6625
Epoch 008  Val AUC 0.6695
Epoch 009  Val AUC 0.6803
Epoch 010  Val AUC 0.6914
Epoch 011  Val AUC 0.7030
Epoch 012  Val AUC 0.6666
Epoch 013  Val AUC 0.6549
Epoch 014  Val AUC 0.6657
Epoch 015  Val AUC 0.6782
Epoch 016  Val AUC 0.6798
Epoch 017  Val AUC 0.7172
Epoch 018  Val AUC 0.7076
Epoch 019  Val AUC 0.7402
Epoch 020  Val AUC 0.7664
Epoch 021  Val AUC 0.7314
Epoch 022  Val AUC 0.7506
Epoch 023  Val AUC 0.7520
Epoch 024  Val AUC 0.7447
Epoch 025  Val AUC 0.7949
Epoch 026  Val AUC 0.7596
Epoch 027  Val AUC 0.7423
Epoch 028  Val AUC 0.7261
Epoch 029  Val AUC 0.7291
Epoch 030  Val AUC 0.7259
Epoch 031  Val AUC 0.7162
Epoch 032  Val AUC 0.7532
Epoch 033  Val AUC 0.7410
Epoch 034  Val AUC 0.7313
Epoch 

# Hierarchical MLP architecture for AMR prediction

We implement a hierarchical neural network pipeline for antimicrobial
resistance prediction from MALDI-TOF spectra.

The system consists of three stages:

Stage 1 — Binary classifier  
Detect multi-resistant isolates.

Stage 2 — Multiclass classifier  
Predict the exact resistance pattern among multi-resistant isolates.

Stage 3 — Binary classifiers  
For non-multi isolates, one binary classifier per antibiotic predicts
resistance independently.

Predictions from all stages are combined to reconstruct the final
resistance vector for each isolate.

The system is trained **per species** using balanced subsets of the data
to prevent bias towards susceptible isolates.

Evaluation metrics:

• Weighted F1 score  
• Accuracy  
• Hamming loss  
• ROC-AUC

In [59]:
class MLP(nn.Module):

    def __init__(self, output_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(6000,512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),

            nn.Linear(512,256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),

            nn.Linear(256,128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128,64),
            nn.GELU(),
            nn.LayerNorm(64),
            nn.Dropout(0.2),

            nn.Linear(64,output_dim)
        )

    def forward(self,x):

        out = self.net(x)

        if out.shape[1] == 1:
            out = out.squeeze(1)

        return out

In [57]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device):

    best_auc = -np.inf
    patience = 25
    epochs_no_improve = 0
    best_model = None

    for epoch in range(200):

        model.train()

        for Xb,yb in train_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(Xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

        model.eval()

        preds = []
        targets = []

        with torch.no_grad():

            for Xb,yb in val_loader:

                Xb = Xb.to(device)

                logits = model(Xb)

                probs = torch.sigmoid(logits).cpu().numpy()

                preds.extend(probs)
                targets.extend(yb.numpy())

        auc = roc_auc_score(targets,preds)

        if auc > best_auc:

            best_auc = auc
            best_model = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            break

    model.load_state_dict(best_model)

    return model

In [60]:
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, roc_auc_score

results = {}

for species, ab_list in species_antibiotics.items():

    print("\n==============================")
    print("SPECIES:",species)
    print("==============================")

    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask = y_all == species

    X_species = X_all[mask]
    amr_species = amr_all[mask][:,idx_ab]

    resistance_counts = amr_species.sum(axis=1)

    y_multi = (resistance_counts > 1).astype(int)

    # Stage 1 model
    model_stage1 = MLP(1).to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(model_stage1.parameters(),lr=3e-4)

    # train stage 1
    model_stage1 = train_model(
        model_stage1,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        device
    )

    # predictions stage 1
    multi_pred = []

    model_stage1.eval()

    with torch.no_grad():

        logits = model_stage1(torch.tensor(X_species,dtype=torch.float32).to(device))

        multi_pred = (torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int)

    # Stage 2 dataset
    multi_idx = np.where(multi_pred==1)[0]

    X_multi = X_species[multi_idx]

    patterns = [tuple(row) for row in amr_species[multi_idx]]

    pattern_to_class = {p:i for i,p in enumerate(set(patterns))}

    y_lps = np.array([pattern_to_class[p] for p in patterns])

    # Stage 3 dataset
    non_multi_idx = np.where(multi_pred==0)[0]

    X_non_multi = X_species[non_multi_idx]
    y_non_multi = amr_species[non_multi_idx]

    # Binary classifiers per antibiotic
    ab_models = []

    for j in range(len(ab_list)):

        y_ab = y_non_multi[:,j]

        model_ab = MLP(1).to(device)

        optimizer = torch.optim.AdamW(model_ab.parameters(),lr=3e-4)

        model_ab = train_model(
            model_ab,
            train_loader,
            val_loader,
            nn.BCEWithLogitsLoss(),
            optimizer,
            device
        )

        ab_models.append(model_ab)


SPECIES: Staphylococcus_Aureus


RuntimeError: result type Float can't be cast to the desired output type Long

In [45]:
pred_matrix = np.zeros_like(amr_species)

# multi predictions
for i,idx in enumerate(multi_idx):

    pattern = list(pattern_to_class.keys())[y_lps[i]]

    pred_matrix[idx] = pattern

# single predictions
for j,model in enumerate(ab_models):

    with torch.no_grad():

        logits = model(
            torch.tensor(X_non_multi,dtype=torch.float32).to(device)
        )

        preds = (torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int)

    pred_matrix[non_multi_idx,j] = preds

NameError: name 'y_lps' is not defined

In [ ]:
true_matrix = amr_species

weighted_f1 = f1_score(
    true_matrix.flatten(),
    pred_matrix.flatten(),
    average="weighted"
)

acc = accuracy_score(
    true_matrix.flatten(),
    pred_matrix.flatten()
)

hamming = hamming_loss(
    true_matrix,
    pred_matrix
)

auc = roc_auc_score(
    true_matrix.flatten(),
    pred_matrix.flatten()
)

print("Weighted F1:",weighted_f1)
print("Accuracy:",acc)
print("Hamming loss:",hamming)
print("AUC:",auc)

In [66]:
from collections import Counter
import copy

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*60)
    print("MULTICLASS TRAINING:", species)
    print("="*60)

    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask_species = y_all == species
    species_indices = np.where(mask_species)[0]

    amr_species = amr_all[species_indices][:, idx_ab]

    resistance_counts = amr_species.sum(axis=1)

    multi_mask = resistance_counts > 1
    multi_indices = species_indices[multi_mask]

    X_multi = X_all[multi_indices]

    patterns = []

    for idx in multi_indices:
        pattern = tuple(amr_all[idx, idx_ab].astype(int))
        patterns.append(pattern)

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    y_multi = np.array([pattern_to_class[p] for p in patterns])

    print("Multi samples:",len(X_multi))
    print("Classes:",len(unique_patterns))

    if len(unique_patterns) < 2:
        print("Skipping species (not enough classes)")
        continue

    # -------------------------
    # train / val / test split
    # -------------------------

    X_train,X_temp,y_train,y_temp = train_test_split(
        X_multi,y_multi,test_size=0.3,stratify=y_multi,random_state=42
    )

    X_val,X_test,y_val,y_test = train_test_split(
        X_temp,y_temp,test_size=0.5,stratify=y_temp,random_state=42
    )

    X_train = torch.tensor(X_train,dtype=torch.float32)
    X_val = torch.tensor(X_val,dtype=torch.float32)
    X_test = torch.tensor(X_test,dtype=torch.float32)

    y_train = torch.tensor(y_train,dtype=torch.long)
    y_val = torch.tensor(y_val,dtype=torch.long)
    y_test = torch.tensor(y_test,dtype=torch.long)

    train_loader = DataLoader(
        TensorDataset(X_train,y_train),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(X_val,y_val),
        batch_size=128
    )

    test_loader = DataLoader(
        TensorDataset(X_test,y_test),
        batch_size=128
    )

    # -------------------------
    # Model
    # -------------------------

    class MultiMLP(nn.Module):

        def __init__(self,n_classes):

            super().__init__()

            self.net = nn.Sequential(

                nn.Linear(6000,512),
                nn.GELU(),
                nn.LayerNorm(512),
                nn.Dropout(0.2),

                nn.Linear(512,256),
                nn.GELU(),
                nn.LayerNorm(256),
                nn.Dropout(0.2),

                nn.Linear(256,128),
                nn.GELU(),
                nn.LayerNorm(128),
                nn.Dropout(0.2),

                nn.Linear(128,64),
                nn.GELU(),
                nn.LayerNorm(64),
                nn.Dropout(0.2),

                nn.Linear(64,n_classes)
            )

        def forward(self,x):
            return self.net(x)

    model = MultiMLP(len(unique_patterns)).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

    # -------------------------
    # Early stopping params
    # -------------------------

    best_auc = -np.inf
    patience = 5
    epochs_no_improve = 0
    best_model = None

    # -------------------------
    # Training loop
    # -------------------------

    for epoch in range(200):

        model.train()

        for Xb,yb in train_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(Xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

        # -------------------------
        # Validation
        # -------------------------

        model.eval()

        preds = []
        targets = []

        with torch.no_grad():

            for Xb,yb in val_loader:

                Xb = Xb.to(device)
                yb = yb.to(device)

                logits = model(Xb)

                probs = torch.softmax(logits,dim=1).cpu().numpy()

                preds.extend(probs)
                targets.extend(yb.cpu().numpy())

        preds = np.array(preds)

        auc = roc_auc_score(targets,preds,multi_class="ovr")

        print(f"Epoch {epoch:03d}  Val AUC {auc:.4f}")

        if auc > best_auc:

            best_auc = auc
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            print("Early stopping triggered")
            break

    model.load_state_dict(best_model)

    print("Best Val AUC:",best_auc)


MULTICLASS TRAINING: Staphylococcus_Aureus
Multi samples: 388
Classes: 4
Epoch 000  Val AUC 0.5493
Epoch 001  Val AUC 0.5727
Epoch 002  Val AUC 0.6213
Epoch 003  Val AUC 0.6502
Epoch 004  Val AUC 0.6725
Epoch 005  Val AUC 0.6908
Epoch 006  Val AUC 0.7188
Epoch 007  Val AUC 0.7490
Epoch 008  Val AUC 0.7681
Epoch 009  Val AUC 0.7701
Epoch 010  Val AUC 0.7546
Epoch 011  Val AUC 0.7302
Epoch 012  Val AUC 0.7198
Epoch 013  Val AUC 0.7397
Epoch 014  Val AUC 0.7336
Early stopping triggered
Best Val AUC: 0.7701189686270056

MULTICLASS TRAINING: Escherichia_Coli
Multi samples: 1116
Classes: 4
Epoch 000  Val AUC 0.6388
Epoch 001  Val AUC 0.6392
Epoch 002  Val AUC 0.6600
Epoch 003  Val AUC 0.6696
Epoch 004  Val AUC 0.6701
Epoch 005  Val AUC 0.6774
Epoch 006  Val AUC 0.6765
Epoch 007  Val AUC 0.6849
Epoch 008  Val AUC 0.7193
Epoch 009  Val AUC 0.7168
Epoch 010  Val AUC 0.7242
Epoch 011  Val AUC 0.7470
Epoch 012  Val AUC 0.7478
Epoch 013  Val AUC 0.7455
Epoch 014  Val AUC 0.7490
Epoch 015  Val AUC

In [67]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

model.eval()

preds = []
targets = []

with torch.no_grad():

    for Xb,yb in test_loader:

        Xb = Xb.to(device)
        yb = yb.to(device)

        logits = model(Xb)

        probs = torch.softmax(logits,dim=1)

        pred_class = torch.argmax(probs,dim=1)

        preds.extend(pred_class.cpu().numpy())
        targets.extend(yb.cpu().numpy())

preds = np.array(preds)
targets = np.array(targets)

print("\n==============================")
print("TEST RESULTS")
print("==============================")

print("Accuracy:",accuracy_score(targets,preds))
print("Macro F1:",f1_score(targets,preds,average="macro"))
print("Weighted F1:",f1_score(targets,preds,average="weighted"))

print("\nConfusion Matrix")
print(confusion_matrix(targets,preds))

print("\nClassification Report")
print(classification_report(targets,preds))

print("\nClass ↔ Resistance Pattern Mapping")

for pattern,cls in pattern_to_class.items():

    sr_pattern = ["R" if x==1 else "S" for x in pattern]

    print(f"class {cls} → {' '.join(sr_pattern)}")


TEST RESULTS
Accuracy: 0.7323232323232324
Macro F1: 0.5869107744107743
Weighted F1: 0.6855582338536884

Confusion Matrix
[[ 12  11   2]
 [  2 121   0]
 [  5  33  12]]

Classification Report
              precision    recall  f1-score   support

           0       0.63      0.48      0.55        25
           1       0.73      0.98      0.84       123
           2       0.86      0.24      0.38        50

    accuracy                           0.73       198
   macro avg       0.74      0.57      0.59       198
weighted avg       0.75      0.73      0.69       198


Class ↔ Resistance Pattern Mapping
class 0 → S R R
class 1 → R R S
class 2 → R R R


# Hierarchical MLP tree for AMR prediction from MALDI-TOF spectra

We implement a species-specific hierarchical neural network architecture for AMR prediction.

For each bacterial species, the pipeline is:

1. **Stage 1 — Binary classifier**
   Detect whether an isolate is **multi-resistant** or **non-multi-resistant**.

2. **Stage 2 — Multiclass classifier**
   For isolates predicted as multi-resistant, classify the **exact LPS resistance pattern**.

3. **Stage 3 — Binary classifiers**
   For isolates predicted as non-multi-resistant, train one binary classifier per antibiotic.

4. **Prediction reconstruction**
   The outputs from Stage 2 and Stage 3 are merged to reconstruct the final resistance vector.

5. **Evaluation**
   Final predictions are evaluated as a multi-label problem using:
   - Weighted F1
   - Accuracy
   - Hamming loss
   - ROC-AUC

To avoid strong class imbalance, for each species we first construct a balanced subset:
- keep all **single-resistant** isolates
- keep all **multi-resistant** isolates
- sample only a fraction of **susceptible** isolates

In [69]:
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    hamming_loss,
    roc_auc_score
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

SEED = 42
ALPHA = 1.5
BATCH_SIZE = 128
MAX_EPOCHS = 200
PATIENCE = 25
LR = 3e-4

rng = np.random.default_rng(SEED)

# We assume these already exist:
# X_all, y_all, amr_all, species_antibiotics, antibiotic_index

Using device: cpu


# Base MLP architecture

We use the MLP size M described in the paper:

6000 → 512 → 256 → 128 → 64

Between layers:
- GeLU
- LayerNorm
- Dropout(0.2)

For binary tasks the network outputs one logit.
For multiclass tasks it outputs one logit per class.

In [70]:
class MLP(nn.Module):
    def __init__(self, output_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(6000, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.GELU(),
            nn.LayerNorm(64),
            nn.Dropout(0.2),

            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        out = self.net(x)
        if out.ndim == 2 and out.shape[1] == 1:
            out = out.squeeze(1)
        return out

# Utility functions

These helper functions handle:

- balanced subset creation
- safe train/validation/test split
- DataLoader creation
- model training with early stopping
- prediction for binary and multiclass tasks
- metric computation

In [71]:
def can_stratify(y):
    values, counts = np.unique(y, return_counts=True)
    return len(values) > 1 and np.min(counts) >= 2


def safe_split_indices(y, test_size=0.30, random_state=42):
    idx = np.arange(len(y))
    strat = y if can_stratify(y) else None
    idx_train, idx_temp = train_test_split(
        idx,
        test_size=test_size,
        stratify=strat,
        random_state=random_state
    )

    y_temp = y[idx_temp]
    strat_temp = y_temp if can_stratify(y_temp) else None

    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=0.50,
        stratify=strat_temp,
        random_state=random_state
    )
    return idx_train, idx_val, idx_test


def make_tensor_loader(X, y, batch_size=128, shuffle=False, y_dtype=torch.float32):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=y_dtype)
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def make_feature_loader(X, batch_size=128):
    X_t = torch.tensor(X, dtype=torch.float32)
    ds = TensorDataset(X_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


def make_balanced_subset_for_species(species, alpha=1.5, seed=42):
    local_rng = np.random.default_rng(seed)

    ab_list = species_antibiotics[species]
    idx_ab = [antibiotic_index[a] for a in ab_list]

    mask_species = (y_all == species)
    species_indices = np.where(mask_species)[0]

    y_matrix = amr_all[species_indices][:, idx_ab]
    resistance_counts = y_matrix.sum(axis=1)

    susceptible_idx_local = np.where(resistance_counts == 0)[0]
    single_idx_local = np.where(resistance_counts == 1)[0]
    multi_idx_local = np.where(resistance_counts > 1)[0]

    n_resistant = len(single_idx_local) + len(multi_idx_local)
    n_sus_target = int(alpha * n_resistant)

    if len(susceptible_idx_local) > n_sus_target:
        susceptible_sample_local = local_rng.choice(
            susceptible_idx_local,
            size=n_sus_target,
            replace=False
        )
    else:
        susceptible_sample_local = susceptible_idx_local

    selected_local = np.concatenate([
        single_idx_local,
        multi_idx_local,
        susceptible_sample_local
    ])

    X_species = X_all[species_indices][selected_local]
    Y_species = y_matrix[selected_local]

    return X_species, Y_species, ab_list


def train_binary_model(X_train, y_train, X_val, y_val,
                       lr=3e-4, batch_size=128, max_epochs=200,
                       patience=25, device="cpu"):
    unique_train = np.unique(y_train)

    if len(unique_train) == 1:
        constant_value = int(unique_train[0])
        return {
            "kind": "constant",
            "value": constant_value
        }

    train_loader = make_tensor_loader(
        X_train, y_train, batch_size=batch_size, shuffle=True, y_dtype=torch.float32
    )
    val_loader = make_tensor_loader(
        X_val, y_val, batch_size=batch_size, shuffle=False, y_dtype=torch.float32
    )

    model = MLP(output_dim=1).to(device)

    n_pos = np.sum(y_train == 1)
    n_neg = np.sum(y_train == 0)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_state = None
    best_score = -np.inf
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb = Xb.to(device)
            yb = yb.float().to(device)

            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_probs = []
        val_targets = []
        val_loss_sum = 0.0
        val_n = 0

        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device)
                yb = yb.float().to(device)

                logits = model(Xb)
                loss = criterion(logits, yb)

                probs = torch.sigmoid(logits)

                val_probs.extend(probs.cpu().numpy())
                val_targets.extend(yb.cpu().numpy())

                val_loss_sum += loss.item() * len(yb)
                val_n += len(yb)

        val_targets = np.array(val_targets)
        val_probs = np.array(val_probs)
        val_loss = val_loss_sum / max(val_n, 1)

        if len(np.unique(val_targets)) > 1:
            score = roc_auc_score(val_targets, val_probs)
        else:
            score = -val_loss

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    model.load_state_dict(best_state)

    return {
        "kind": "mlp",
        "model": model
    }


def predict_binary_model(predictor, X, batch_size=128, device="cpu"):
    if len(X) == 0:
        return np.array([]), np.array([])

    if predictor["kind"] == "constant":
        value = predictor["value"]
        probs = np.full(len(X), float(value))
        preds = np.full(len(X), int(value))
        return probs, preds

    model = predictor["model"]
    model.eval()

    loader = make_feature_loader(X, batch_size=batch_size)
    probs = []

    with torch.no_grad():
        for (Xb,) in loader:
            Xb = Xb.to(device)
            logits = model(Xb)
            batch_probs = torch.sigmoid(logits).cpu().numpy()
            probs.extend(batch_probs)

    probs = np.array(probs)
    preds = (probs >= 0.5).astype(int)
    return probs, preds


def train_multiclass_model(X_train, y_train, X_val, y_val, n_classes,
                           lr=3e-4, batch_size=128, max_epochs=200,
                           patience=25, device="cpu"):
    if len(y_train) == 0:
        return {
            "kind": "absent",
            "class_id": 0,
            "n_classes": n_classes
        }

    unique_train = np.unique(y_train)

    if len(unique_train) == 1:
        return {
            "kind": "constant",
            "class_id": int(unique_train[0]),
            "n_classes": n_classes
        }

    train_loader = make_tensor_loader(
        X_train, y_train, batch_size=batch_size, shuffle=True, y_dtype=torch.long
    )
    val_loader = make_tensor_loader(
        X_val, y_val, batch_size=batch_size, shuffle=False, y_dtype=torch.long
    )

    model = MLP(output_dim=n_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    best_state = None
    best_score = np.inf
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb = Xb.to(device)
            yb = yb.long().to(device)

            optimizer.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss_sum = 0.0
        val_n = 0

        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device)
                yb = yb.long().to(device)

                logits = model(Xb)
                loss = criterion(logits, yb)

                val_loss_sum += loss.item() * len(yb)
                val_n += len(yb)

        val_loss = val_loss_sum / max(val_n, 1)

        if val_loss < best_score:
            best_score = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    model.load_state_dict(best_state)

    return {
        "kind": "mlp",
        "model": model,
        "n_classes": n_classes
    }


def predict_multiclass_model(predictor, X, batch_size=128, device="cpu"):
    if len(X) == 0:
        n_classes = predictor.get("n_classes", 1)
        return np.zeros((0, n_classes)), np.array([], dtype=int)

    if predictor["kind"] == "absent":
        n_classes = predictor["n_classes"]
        class_id = predictor["class_id"]
        probs = np.zeros((len(X), n_classes))
        probs[:, class_id] = 1.0
        preds = np.full(len(X), class_id, dtype=int)
        return probs, preds

    if predictor["kind"] == "constant":
        n_classes = predictor["n_classes"]
        class_id = predictor["class_id"]
        probs = np.zeros((len(X), n_classes))
        probs[:, class_id] = 1.0
        preds = np.full(len(X), class_id, dtype=int)
        return probs, preds

    model = predictor["model"]
    model.eval()

    loader = make_feature_loader(X, batch_size=batch_size)
    probs = []

    with torch.no_grad():
        for (Xb,) in loader:
            Xb = Xb.to(device)
            logits = model(Xb)
            batch_probs = torch.softmax(logits, dim=1).cpu().numpy()
            probs.append(batch_probs)

    probs = np.vstack(probs)
    preds = np.argmax(probs, axis=1)
    return probs, preds


def safe_auc_binary(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def compute_final_metrics(true_matrix, pred_matrix, prob_matrix):
    metrics = {}

    metrics["weighted_f1"] = f1_score(
        true_matrix.flatten(),
        pred_matrix.flatten(),
        average="weighted",
        zero_division=0
    )

    metrics["accuracy"] = accuracy_score(
        true_matrix.flatten(),
        pred_matrix.flatten()
    )

    metrics["hamming_loss"] = hamming_loss(
        true_matrix,
        pred_matrix
    )

    if len(np.unique(true_matrix.flatten())) > 1:
        metrics["auc"] = roc_auc_score(
            true_matrix.flatten(),
            prob_matrix.flatten()
        )
    else:
        metrics["auc"] = np.nan

    return metrics

# Full hierarchical training and evaluation loop

For each species:

1. build the balanced subset
2. split into train / validation / test
3. train Stage 1 (multi vs rest)
4. train Stage 2 on true multi-resistant training samples
5. train Stage 3 on true non-multi training samples
6. run the full hierarchical pipeline on the test set
7. reconstruct the final resistance vector
8. compute species-level and antibiotic-level metrics

In [72]:
species_results = []
antibiotic_results = []

for species in species_antibiotics.keys():

    print("\n" + "=" * 80)
    print("SPECIES:", species)
    print("=" * 80)

    # --------------------------------------------------
    # Balanced subset for this species
    # --------------------------------------------------
    X_species, Y_species, ab_list = make_balanced_subset_for_species(
        species,
        alpha=ALPHA,
        seed=SEED
    )

    n_samples = len(X_species)
    n_antibiotics = Y_species.shape[1]

    y_stage1 = (Y_species.sum(axis=1) > 1).astype(int)

    print("Balanced subset shape:", X_species.shape)
    print("Antibiotics:", ab_list)
    print("Non-multi:", np.sum(y_stage1 == 0))
    print("Multi:", np.sum(y_stage1 == 1))

    # --------------------------------------------------
    # Split once at isolate level
    # --------------------------------------------------
    idx_train, idx_val, idx_test = safe_split_indices(
        y_stage1,
        test_size=0.30,
        random_state=SEED
    )

    X_train = X_species[idx_train]
    X_val = X_species[idx_val]
    X_test = X_species[idx_test]

    Y_train = Y_species[idx_train]
    Y_val = Y_species[idx_val]
    Y_test = Y_species[idx_test]

    y1_train = y_stage1[idx_train]
    y1_val = y_stage1[idx_val]
    y1_test = y_stage1[idx_test]

    # --------------------------------------------------
    # Stage 1: multi vs rest
    # --------------------------------------------------
    print("\nTraining Stage 1: multi vs non-multi")

    stage1_predictor = train_binary_model(
        X_train, y1_train,
        X_val, y1_val,
        lr=LR,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=device
    )

    # --------------------------------------------------
    # Stage 2: multiclass LPS among true multi samples
    # --------------------------------------------------
    print("Training Stage 2: multiclass LPS among true multi samples")

    train_multi_mask = (y1_train == 1)
    val_multi_mask = (y1_val == 1)

    X2_train = X_train[train_multi_mask]
    X2_val = X_val[val_multi_mask]

    Y2_train = Y_train[train_multi_mask]
    Y2_val = Y_val[val_multi_mask]

    all_multi_patterns = sorted({
        tuple(row.astype(int)) for row in Y_species[y_stage1 == 1]
    })

    if len(all_multi_patterns) == 0:
        all_multi_patterns = [tuple([0] * n_antibiotics)]

    pattern_to_class = {
        pattern: i for i, pattern in enumerate(all_multi_patterns)
    }
    class_to_pattern = {
        i: np.array(pattern, dtype=int) for pattern, i in pattern_to_class.items()
    }
    patterns_array = np.array(
        [class_to_pattern[i] for i in range(len(class_to_pattern))],
        dtype=int
    )

    y2_train = np.array([
        pattern_to_class[tuple(row.astype(int))] for row in Y2_train
    ]) if len(Y2_train) > 0 else np.array([])

    y2_val = np.array([
        pattern_to_class[tuple(row.astype(int))] for row in Y2_val
    ]) if len(Y2_val) > 0 else np.array([])

    stage2_predictor = train_multiclass_model(
        X2_train, y2_train,
        X2_val, y2_val,
        n_classes=len(pattern_to_class),
        lr=LR,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=device
    )

    # --------------------------------------------------
    # Stage 3: one binary classifier per antibiotic
    # --------------------------------------------------
    print("Training Stage 3: one binary classifier per antibiotic")

    train_non_multi_mask = (y1_train == 0)
    val_non_multi_mask = (y1_val == 0)

    X3_train = X_train[train_non_multi_mask]
    X3_val = X_val[val_non_multi_mask]

    Y3_train = Y_train[train_non_multi_mask]
    Y3_val = Y_val[val_non_multi_mask]

    stage3_predictors = {}

    for j, antibiotic in enumerate(ab_list):
        y3_train_ab = Y3_train[:, j] if len(Y3_train) > 0 else np.array([])
        y3_val_ab = Y3_val[:, j] if len(Y3_val) > 0 else np.array([])

        predictor_ab = train_binary_model(
            X3_train, y3_train_ab,
            X3_val, y3_val_ab,
            lr=LR,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            device=device
        )

        stage3_predictors[antibiotic] = predictor_ab

    # --------------------------------------------------
    # Full hierarchical inference on test set
    # --------------------------------------------------
    print("Running full hierarchical inference on test set")

    stage1_probs_test, stage1_preds_test = predict_binary_model(
        stage1_predictor,
        X_test,
        batch_size=BATCH_SIZE,
        device=device
    )

    pred_matrix = np.zeros_like(Y_test, dtype=int)
    prob_matrix = np.zeros_like(Y_test, dtype=float)

    pred_multi_mask = (stage1_preds_test == 1)
    pred_non_multi_mask = (stage1_preds_test == 0)

    # Stage 2 branch
    if np.sum(pred_multi_mask) > 0:
        X_test_multi_branch = X_test[pred_multi_mask]

        stage2_probs_test, stage2_preds_test = predict_multiclass_model(
            stage2_predictor,
            X_test_multi_branch,
            batch_size=BATCH_SIZE,
            device=device
        )

        pred_patterns = patterns_array[stage2_preds_test]
        pred_matrix[pred_multi_mask] = pred_patterns

        # Convert class probabilities to per-antibiotic probabilities
        prob_patterns = stage2_probs_test @ patterns_array
        prob_matrix[pred_multi_mask] = prob_patterns

    # Stage 3 branch
    if np.sum(pred_non_multi_mask) > 0:
        X_test_non_multi_branch = X_test[pred_non_multi_mask]

        for j, antibiotic in enumerate(ab_list):
            probs_ab, preds_ab = predict_binary_model(
                stage3_predictors[antibiotic],
                X_test_non_multi_branch,
                batch_size=BATCH_SIZE,
                device=device
            )

            prob_matrix[pred_non_multi_mask, j] = probs_ab
            pred_matrix[pred_non_multi_mask, j] = preds_ab

    # --------------------------------------------------
    # Final metrics
    # --------------------------------------------------
    metrics = compute_final_metrics(
        true_matrix=Y_test,
        pred_matrix=pred_matrix,
        prob_matrix=prob_matrix
    )

    print("\nFinal hierarchical metrics")
    print("Weighted F1 :", round(metrics["weighted_f1"], 4))
    print("Accuracy    :", round(metrics["accuracy"], 4))
    print("Hamming loss:", round(metrics["hamming_loss"], 4))
    print("AUC         :", round(metrics["auc"], 4) if not np.isnan(metrics["auc"]) else np.nan)

    species_results.append({
        "species": species,
        "n_samples_balanced": n_samples,
        "n_train": len(X_train),
        "n_val": len(X_val),
        "n_test": len(X_test),
        "n_antibiotics": n_antibiotics,
        "antibiotics": ab_list,
        "weighted_f1": metrics["weighted_f1"],
        "accuracy": metrics["accuracy"],
        "hamming_loss": metrics["hamming_loss"],
        "auc": metrics["auc"]
    })

    # --------------------------------------------------
    # Antibiotic-level metrics
    # --------------------------------------------------
    for j, antibiotic in enumerate(ab_list):
        y_true_ab = Y_test[:, j]
        y_pred_ab = pred_matrix[:, j]
        y_prob_ab = prob_matrix[:, j]

        wf1_ab = f1_score(
            y_true_ab,
            y_pred_ab,
            average="weighted",
            zero_division=0
        )
        acc_ab = accuracy_score(y_true_ab, y_pred_ab)
        hl_ab = hamming_loss(y_true_ab, y_pred_ab)
        auc_ab = safe_auc_binary(y_true_ab, y_prob_ab)

        antibiotic_results.append({
            "species": species,
            "antibiotic": antibiotic,
            "weighted_f1": wf1_ab,
            "accuracy": acc_ab,
            "hamming_loss": hl_ab,
            "auc": auc_ab
        })


SPECIES: Staphylococcus_Aureus
Balanced subset shape: (3472, 6000)
Antibiotics: ['Oxacillin', 'Clindamycin', 'Fusidic acid']
Non-multi: 3084
Multi: 388

Training Stage 1: multi vs non-multi
Training Stage 2: multiclass LPS among true multi samples
Training Stage 3: one binary classifier per antibiotic
Running full hierarchical inference on test set

Final hierarchical metrics
Weighted F1 : 0.8174
Accuracy    : 0.8042
Hamming loss: 0.1958
AUC         : 0.8042

SPECIES: Escherichia_Coli
Balanced subset shape: (4750, 6000)
Antibiotics: ['Ciprofloxacin', 'Ceftriaxone', 'Cefepime']
Non-multi: 3634
Multi: 1116

Training Stage 1: multi vs non-multi
Training Stage 2: multiclass LPS among true multi samples
Training Stage 3: one binary classifier per antibiotic
Running full hierarchical inference on test set

Final hierarchical metrics
Weighted F1 : 0.8143
Accuracy    : 0.8116
Hamming loss: 0.1884
AUC         : 0.8173

SPECIES: Klebsiella_Pneumoniae
Balanced subset shape: (12000, 6000)
Antibio

In [73]:
species_results_df = pd.DataFrame(species_results)
antibiotic_results_df = pd.DataFrame(antibiotic_results)

print("\nSpecies-level results")
display(species_results_df)

print("\nAntibiotic-level results")
display(antibiotic_results_df)


Species-level results


,species,n_samples_balanced,n_train,n_val,n_test,n_antibiotics,antibiotics,weighted_f1,accuracy,hamming_loss,auc
0,Staphylococcus_Aureus,3472,2430,521,521,3,"[Oxacillin, Clindamycin, Fusidic acid]",0.817384,0.804223,0.195777,0.804165
1,Escherichia_Coli,4750,3325,712,713,3,"[Ciprofloxacin, Ceftriaxone, Cefepime]",0.814345,0.811594,0.188406,0.817348
2,Klebsiella_Pneumoniae,12000,8400,1800,1800,3,"[Ciprofloxacin, Imipenem, Meropenem]",0.841540,0.828148,0.171852,0.849598
3,Pseudomonas_Aeruginosa,3848,2693,577,578,3,"[Ciprofloxacin, Imipenem, Meropenem]",0.836749,0.836794,0.163206,0.829087



Antibiotic-level results


,species,antibiotic,weighted_f1,accuracy,hamming_loss,auc
0,Staphylococcus_Aureus,Oxacillin,0.823422,0.815739,0.184261,0.858586
1,Staphylococcus_Aureus,Clindamycin,0.710831,0.677543,0.322457,0.737886
2,Staphylococcus_Aureus,Fusidic acid,0.907037,0.919386,0.080614,0.725521
3,Escherichia_Coli,Ciprofloxacin,0.719912,0.715288,0.284712,0.795259
4,Escherichia_Coli,Ceftriaxone,0.855497,0.856942,0.143058,0.816952
5,Escherichia_Coli,Cefepime,0.864816,0.862553,0.137447,0.815443
6,Klebsiella_Pneumoniae,Ciprofloxacin,0.691584,0.687222,0.312778,0.763253
7,Klebsiella_Pneumoniae,Imipenem,0.901985,0.879444,0.120556,0.642366
8,Klebsiella_Pneumoniae,Meropenem,0.932321,0.917778,0.082222,0.704489
9,Pseudomonas_Aeruginosa,Ciprofloxacin,0.815173,0.814879,0.185121,0.823507


In [74]:
species_results_df = species_results_df.sort_values("species").reset_index(drop=True)
antibiotic_results_df = antibiotic_results_df.sort_values(
    ["species", "antibiotic"]
).reset_index(drop=True)

display(species_results_df)
display(antibiotic_results_df)

,species,n_samples_balanced,n_train,n_val,n_test,n_antibiotics,antibiotics,weighted_f1,accuracy,hamming_loss,auc
0,Escherichia_Coli,4750,3325,712,713,3,"[Ciprofloxacin, Ceftriaxone, Cefepime]",0.814345,0.811594,0.188406,0.817348
1,Klebsiella_Pneumoniae,12000,8400,1800,1800,3,"[Ciprofloxacin, Imipenem, Meropenem]",0.841540,0.828148,0.171852,0.849598
2,Pseudomonas_Aeruginosa,3848,2693,577,578,3,"[Ciprofloxacin, Imipenem, Meropenem]",0.836749,0.836794,0.163206,0.829087
3,Staphylococcus_Aureus,3472,2430,521,521,3,"[Oxacillin, Clindamycin, Fusidic acid]",0.817384,0.804223,0.195777,0.804165


,species,antibiotic,weighted_f1,accuracy,hamming_loss,auc
0,Escherichia_Coli,Cefepime,0.864816,0.862553,0.137447,0.815443
1,Escherichia_Coli,Ceftriaxone,0.855497,0.856942,0.143058,0.816952
2,Escherichia_Coli,Ciprofloxacin,0.719912,0.715288,0.284712,0.795259
3,Klebsiella_Pneumoniae,Ciprofloxacin,0.691584,0.687222,0.312778,0.763253
4,Klebsiella_Pneumoniae,Imipenem,0.901985,0.879444,0.120556,0.642366
5,Klebsiella_Pneumoniae,Meropenem,0.932321,0.917778,0.082222,0.704489
6,Pseudomonas_Aeruginosa,Ciprofloxacin,0.815173,0.814879,0.185121,0.823507
7,Pseudomonas_Aeruginosa,Imipenem,0.893904,0.894464,0.105536,0.883229
8,Pseudomonas_Aeruginosa,Meropenem,0.810151,0.801038,0.198962,0.534844
9,Staphylococcus_Aureus,Clindamycin,0.710831,0.677543,0.322457,0.737886


In [75]:
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, roc_auc_score

lps_species_results = []
lps_antibiotic_results = []

for species in species_antibiotics.keys():

    print("\n"+"="*80)
    print("LPS BENCHMARK:",species)
    print("="*80)

    # ----------------------------
    # balanced subset
    # ----------------------------

    X_species, Y_species, ab_list = make_balanced_subset_for_species(
        species,
        alpha=ALPHA,
        seed=SEED
    )

    n_antibiotics = Y_species.shape[1]

    # ----------------------------
    # build LPS classes
    # ----------------------------

    patterns = [tuple(row.astype(int)) for row in Y_species]

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {
        p:i for i,p in enumerate(unique_patterns)
    }

    class_to_pattern = {
        i:np.array(p) for p,i in pattern_to_class.items()
    }

    y_lps = np.array([pattern_to_class[p] for p in patterns])

    print("Samples:",len(X_species))
    print("LPS classes:",len(unique_patterns))

    # ----------------------------
    # split
    # ----------------------------

    idx_train,idx_val,idx_test = safe_split_indices(
        y_lps,
        test_size=0.30,
        random_state=SEED
    )

    X_train = X_species[idx_train]
    X_val = X_species[idx_val]
    X_test = X_species[idx_test]

    y_train = y_lps[idx_train]
    y_val = y_lps[idx_val]
    y_test = y_lps[idx_test]

    Y_test = Y_species[idx_test]

    # ----------------------------
    # train multiclass model
    # ----------------------------

    predictor = train_multiclass_model(
        X_train,
        y_train,
        X_val,
        y_val,
        n_classes=len(unique_patterns),
        lr=LR,
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        device=device
    )

    # ----------------------------
    # prediction
    # ----------------------------

    probs, preds = predict_multiclass_model(
        predictor,
        X_test,
        batch_size=BATCH_SIZE,
        device=device
    )

    pred_patterns = np.array([
        class_to_pattern[c] for c in preds
    ])

    # convert probabilities to antibiotic probabilities

    patterns_matrix = np.array([
        class_to_pattern[i] for i in range(len(unique_patterns))
    ])

    prob_matrix = probs @ patterns_matrix

    # ----------------------------
    # metrics
    # ----------------------------

    metrics = compute_final_metrics(
        true_matrix=Y_test,
        pred_matrix=pred_patterns,
        prob_matrix=prob_matrix
    )

    print("\nLPS benchmark metrics")
    print("Weighted F1:",metrics["weighted_f1"])
    print("Accuracy:",metrics["accuracy"])
    print("Hamming loss:",metrics["hamming_loss"])
    print("AUC:",metrics["auc"])

    lps_species_results.append({
        "species":species,
        "weighted_f1":metrics["weighted_f1"],
        "accuracy":metrics["accuracy"],
        "hamming_loss":metrics["hamming_loss"],
        "auc":metrics["auc"],
        "n_classes":len(unique_patterns)
    })

    # ----------------------------
    # antibiotic metrics
    # ----------------------------

    for j,antibiotic in enumerate(ab_list):

        y_true = Y_test[:,j]
        y_pred = pred_patterns[:,j]
        y_prob = prob_matrix[:,j]

        wf1 = f1_score(y_true,y_pred,average="weighted",zero_division=0)
        acc = accuracy_score(y_true,y_pred)
        hl = hamming_loss(y_true,y_pred)

        if len(np.unique(y_true))>1:
            auc = roc_auc_score(y_true,y_prob)
        else:
            auc = np.nan

        lps_antibiotic_results.append({
            "species":species,
            "antibiotic":antibiotic,
            "weighted_f1":wf1,
            "accuracy":acc,
            "hamming_loss":hl,
            "auc":auc
        })


LPS BENCHMARK: Staphylococcus_Aureus
Samples: 3472
LPS classes: 8

LPS benchmark metrics
Weighted F1: 0.8533750454309361
Accuracy: 0.8784388995521433
Hamming loss: 0.12156110044785669
AUC: 0.822732090172152

LPS BENCHMARK: Escherichia_Coli
Samples: 4750
LPS classes: 8

LPS benchmark metrics
Weighted F1: 0.780636712738626
Accuracy: 0.7947639083683965
Hamming loss: 0.20523609163160356
AUC: 0.7965753254636828

LPS BENCHMARK: Klebsiella_Pneumoniae
Samples: 12000
LPS classes: 8

LPS benchmark metrics
Weighted F1: 0.8764943128172056
Accuracy: 0.8807407407407407
Hamming loss: 0.11925925925925926
AUC: 0.8950311468219063

LPS BENCHMARK: Pseudomonas_Aeruginosa
Samples: 3848
LPS classes: 7

LPS benchmark metrics
Weighted F1: 0.8680522487946156
Accuracy: 0.8754325259515571
Hamming loss: 0.1245674740484429
AUC: 0.8596810897817107


In [76]:
class MultitaskMLP(nn.Module):

    def __init__(self, n_antibiotics, n_lps_classes):

        super().__init__()

        # shared embedding
        self.encoder = nn.Sequential(

            nn.Linear(6000,512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),

            nn.Linear(512,256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),

            nn.Linear(256,128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128,64),
            nn.GELU(),
            nn.LayerNorm(64),
        )

        # heads

        self.lps_head = nn.Linear(64,n_lps_classes)

        self.ab_heads = nn.Linear(64,n_antibiotics)

        self.multi_head = nn.Linear(64,1)

    def forward(self,x):

        emb = self.encoder(x)

        lps_logits = self.lps_head(emb)

        ab_logits = self.ab_heads(emb)

        multi_logits = self.multi_head(emb).squeeze(1)

        return lps_logits,ab_logits,multi_logits,emb

In [78]:
class MultiTaskMLP(nn.Module):

    def __init__(self, n_lps_classes, n_antibiotics):

        super().__init__()

        # shared embedding
        self.embedding = nn.Sequential(

            nn.Linear(6000,512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),

            nn.Linear(512,256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),

            nn.Linear(256,128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128,64),
            nn.GELU(),
            nn.LayerNorm(64)
        )

        # LPS head
        self.lps_head = nn.Linear(64,n_lps_classes)

        # antibiotic heads
        self.ab_heads = nn.ModuleList([
            nn.Linear(64,1) for _ in range(n_antibiotics)
        ])

        # multi resistance detection
        self.multi_head = nn.Linear(64,1)

    def forward(self,x):

        emb = self.embedding(x)

        lps_logits = self.lps_head(emb)

        ab_logits = [head(emb).squeeze(1) for head in self.ab_heads]

        multi_logits = self.multi_head(emb).squeeze(1)

        return lps_logits, ab_logits, multi_logits

In [79]:
patterns = [tuple(row.astype(int)) for row in Y_species]

unique_patterns = sorted(list(set(patterns)))

pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

class_to_pattern = {i:p for p,i in pattern_to_class.items()}

y_lps = np.array([pattern_to_class[p] for p in patterns])

y_multi = (Y_species.sum(axis=1) > 1).astype(int)

In [80]:
X_train_t = torch.tensor(X_train,dtype=torch.float32)
X_val_t = torch.tensor(X_val,dtype=torch.float32)

y_lps_train_t = torch.tensor(y_lps_train,dtype=torch.long)
y_lps_val_t = torch.tensor(y_lps_val,dtype=torch.long)

y_multi_train_t = torch.tensor(y_multi_train,dtype=torch.float32)
y_multi_val_t = torch.tensor(y_multi_val,dtype=torch.float32)

Y_train_t = torch.tensor(Y_train,dtype=torch.float32)
Y_val_t = torch.tensor(Y_val,dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(
        X_train_t,
        y_lps_train_t,
        Y_train_t,
        y_multi_train_t
    ),
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        X_val_t,
        y_lps_val_t,
        Y_val_t,
        y_multi_val_t
    ),
    batch_size=128
)

In [81]:
model = MultiTaskMLP(
    n_lps_classes=len(unique_patterns),
    n_antibiotics=len(ab_list)
).to(device)

criterion_lps = nn.CrossEntropyLoss()
criterion_binary = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [82]:
best_loss = np.inf
patience = 5
epochs_no_improve = 0
best_model = None

for epoch in range(200):

    model.train()

    for Xb,y_lps_b,Yb,y_multi_b in train_loader:

        Xb = Xb.to(device)
        y_lps_b = y_lps_b.to(device)
        Yb = Yb.to(device)
        y_multi_b = y_multi_b.to(device)

        optimizer.zero_grad()

        lps_logits, ab_logits, multi_logits = model(Xb)

        loss_lps = criterion_lps(lps_logits,y_lps_b)

        loss_ab = 0
        for j in range(len(ab_logits)):
            loss_ab += criterion_binary(
                ab_logits[j],
                Yb[:,j]
            )

        loss_multi = criterion_binary(
            multi_logits,
            y_multi_b
        )

        loss = loss_lps + loss_ab + 0.5*loss_multi

        loss.backward()

        optimizer.step()

    # validation

    model.eval()

    val_loss = 0
    n = 0

    with torch.no_grad():

        for Xb,y_lps_b,Yb,y_multi_b in val_loader:

            Xb = Xb.to(device)
            y_lps_b = y_lps_b.to(device)
            Yb = Yb.to(device)
            y_multi_b = y_multi_b.to(device)

            lps_logits, ab_logits, multi_logits = model(Xb)

            loss_lps = criterion_lps(lps_logits,y_lps_b)

            loss_ab = 0
            for j in range(len(ab_logits)):
                loss_ab += criterion_binary(
                    ab_logits[j],
                    Yb[:,j]
                )

            loss_multi = criterion_binary(
                multi_logits,
                y_multi_b
            )

            loss = loss_lps + loss_ab + 0.5*loss_multi

            val_loss += loss.item()*len(Xb)
            n += len(Xb)

    val_loss /= n

    print("Epoch",epoch,"Val loss",val_loss)

    if val_loss < best_loss:

        best_loss = val_loss
        epochs_no_improve = 0
        best_model = copy.deepcopy(model.state_dict())

    else:

        epochs_no_improve += 1

    if epochs_no_improve >= patience:

        print("Early stopping")
        break

model.load_state_dict(best_model)

Epoch 0 Val loss 2.8463807591092336
Epoch 1 Val loss 2.744747182000393
Epoch 2 Val loss 2.5010730044169067
Epoch 3 Val loss 2.514077913280641
Epoch 4 Val loss 2.33158531298793
Epoch 5 Val loss 2.289936635407285
Epoch 6 Val loss 2.2482804254469624
Epoch 7 Val loss 2.3436307069664952
Epoch 8 Val loss 2.244256503751319
Epoch 9 Val loss 2.242483168325589
Epoch 10 Val loss 2.392357488053774
Epoch 11 Val loss 2.2933248723079513
Epoch 12 Val loss 2.493642742482806
Epoch 13 Val loss 2.361517714904961
Epoch 14 Val loss 2.489662882462573
Early stopping


<All keys matched successfully>

In [83]:
multitask_species_results = []

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*80)
    print("MULTITASK TRAINING:",species)
    print("="*80)

    # ------------------------
    # balanced subset
    # ------------------------

    X_species, Y_species, ab_list = make_balanced_subset_for_species(
        species,
        alpha=ALPHA,
        seed=SEED
    )

    n_antibiotics = Y_species.shape[1]

    # ------------------------
    # build LPS classes
    # ------------------------

    patterns = [tuple(row.astype(int)) for row in Y_species]

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    class_to_pattern = {i:p for p,i in pattern_to_class.items()}

    y_lps = np.array([pattern_to_class[p] for p in patterns])

    y_multi = (Y_species.sum(axis=1) > 1).astype(int)

    # ------------------------
    # split
    # ------------------------

    idx_train,idx_val,idx_test = safe_split_indices(
        y_lps,
        test_size=0.30,
        random_state=SEED
    )

    X_train = X_species[idx_train]
    X_val = X_species[idx_val]
    X_test = X_species[idx_test]

    Y_train = Y_species[idx_train]
    Y_val = Y_species[idx_val]
    Y_test = Y_species[idx_test]

    y_lps_train = y_lps[idx_train]
    y_lps_val = y_lps[idx_val]

    y_multi_train = y_multi[idx_train]
    y_multi_val = y_multi[idx_val]

    # ------------------------
    # tensors
    # ------------------------

    X_train_t = torch.tensor(X_train,dtype=torch.float32)
    X_val_t = torch.tensor(X_val,dtype=torch.float32)

    y_lps_train_t = torch.tensor(y_lps_train,dtype=torch.long)
    y_lps_val_t = torch.tensor(y_lps_val,dtype=torch.long)

    y_multi_train_t = torch.tensor(y_multi_train,dtype=torch.float32)
    y_multi_val_t = torch.tensor(y_multi_val,dtype=torch.float32)

    Y_train_t = torch.tensor(Y_train,dtype=torch.float32)
    Y_val_t = torch.tensor(Y_val,dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(
            X_train_t,
            y_lps_train_t,
            Y_train_t,
            y_multi_train_t
        ),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            X_val_t,
            y_lps_val_t,
            Y_val_t,
            y_multi_val_t
        ),
        batch_size=128
    )

    # ------------------------
    # model
    # ------------------------

    model = MultiTaskMLP(
        n_lps_classes=len(unique_patterns),
        n_antibiotics=n_antibiotics
    ).to(device)

    criterion_lps = nn.CrossEntropyLoss()
    criterion_binary = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4
    )

    # ------------------------
    # training
    # ------------------------

    best_loss = np.inf
    patience = 5
    epochs_no_improve = 0
    best_model = None

    for epoch in range(200):

        model.train()

        for Xb,y_lps_b,Yb,y_multi_b in train_loader:

            Xb = Xb.to(device)
            y_lps_b = y_lps_b.to(device)
            Yb = Yb.to(device)
            y_multi_b = y_multi_b.to(device)

            optimizer.zero_grad()

            lps_logits, ab_logits, multi_logits = model(Xb)

            loss_lps = criterion_lps(lps_logits,y_lps_b)

            loss_ab = 0
            for j in range(len(ab_logits)):
                loss_ab += criterion_binary(
                    ab_logits[j],
                    Yb[:,j]
                )

            loss_multi = criterion_binary(
                multi_logits,
                y_multi_b
            )

            loss = loss_lps + loss_ab + 0.5*loss_multi

            loss.backward()

            optimizer.step()

        # validation

        model.eval()

        val_loss = 0
        n = 0

        with torch.no_grad():

            for Xb,y_lps_b,Yb,y_multi_b in val_loader:

                Xb = Xb.to(device)
                y_lps_b = y_lps_b.to(device)
                Yb = Yb.to(device)
                y_multi_b = y_multi_b.to(device)

                lps_logits, ab_logits, multi_logits = model(Xb)

                loss_lps = criterion_lps(lps_logits,y_lps_b)

                loss_ab = 0
                for j in range(len(ab_logits)):
                    loss_ab += criterion_binary(
                        ab_logits[j],
                        Yb[:,j]
                    )

                loss_multi = criterion_binary(
                    multi_logits,
                    y_multi_b
                )

                loss = loss_lps + loss_ab + 0.5*loss_multi

                val_loss += loss.item()*len(Xb)
                n += len(Xb)

        val_loss /= n

        print("Epoch",epoch,"Val loss",val_loss)

        if val_loss < best_loss:

            best_loss = val_loss
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            print("Early stopping")
            break

    model.load_state_dict(best_model)

    print("Best val loss:",best_loss)

    multitask_species_results.append({
        "species":species,
        "val_loss":best_loss
    })


MULTITASK TRAINING: Staphylococcus_Aureus
Epoch 0 Val loss 2.896940855970767
Epoch 1 Val loss 2.8499302923564946
Epoch 2 Val loss 2.7580833709628934
Epoch 3 Val loss 2.60615605447663
Epoch 4 Val loss 2.398239507784999
Epoch 5 Val loss 2.3402710221779324
Epoch 6 Val loss 2.2685823339875966
Epoch 7 Val loss 2.3102652234902994
Epoch 8 Val loss 2.1720200555109472
Epoch 9 Val loss 2.2387391710922038
Epoch 10 Val loss 2.1456955307504724
Epoch 11 Val loss 2.252795785951523
Epoch 12 Val loss 2.183526699007587
Epoch 13 Val loss 2.248860547272578
Epoch 14 Val loss 2.3322038440008765
Epoch 15 Val loss 2.5999676531427425
Early stopping
Best val loss: 2.1456955307504724

MULTITASK TRAINING: Escherichia_Coli
Epoch 0 Val loss 3.1851278315769154
Epoch 1 Val loss 2.9972359169734997
Epoch 2 Val loss 2.7205541428555264
Epoch 3 Val loss 2.5577302005853544
Epoch 4 Val loss 2.5313457290778
Epoch 5 Val loss 2.7815018396699025
Epoch 6 Val loss 2.5789900602919333
Epoch 7 Val loss 2.534698657775193
Epoch 8 Val

In [84]:
# ----------------------------
# TEST PREDICTIONS
# ----------------------------

X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)

model.eval()

prob_matrix = []
pred_matrix = []

with torch.no_grad():

    lps_logits, ab_logits, multi_logits = model(X_test_t)

    for j in range(len(ab_logits)):

        probs = torch.sigmoid(ab_logits[j]).cpu().numpy()

        preds = (probs > 0.5).astype(int)

        prob_matrix.append(probs)
        pred_matrix.append(preds)

prob_matrix = np.stack(prob_matrix,axis=1)
pred_matrix = np.stack(pred_matrix,axis=1)

# ----------------------------
# METRICS
# ----------------------------

wf1 = f1_score(
    Y_test.flatten(),
    pred_matrix.flatten(),
    average="weighted"
)

acc = accuracy_score(
    Y_test.flatten(),
    pred_matrix.flatten()
)

hl = hamming_loss(
    Y_test,
    pred_matrix
)

auc = roc_auc_score(
    Y_test.flatten(),
    prob_matrix.flatten()
)

print("\nMultitask results")
print("Weighted F1:",wf1)
print("Accuracy:",acc)
print("Hamming loss:",hl)
print("AUC:",auc)

multitask_species_results.append({
    "species":species,
    "weighted_f1":wf1,
    "accuracy":acc,
    "hamming_loss":hl,
    "auc":auc
})


Multitask results
Weighted F1: 0.8679300584069995
Accuracy: 0.8754325259515571
Hamming loss: 0.1245674740484429
AUC: 0.8642609640254685


In [85]:
# --------------------------------------------------
# TEST EVALUATION
# --------------------------------------------------

X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)

model.eval()

prob_matrix = []
pred_matrix = []

with torch.no_grad():

    lps_logits, ab_logits, multi_logits = model(X_test_t)

    for j in range(len(ab_logits)):

        probs = torch.sigmoid(ab_logits[j]).cpu().numpy()
        preds = (probs > 0.5).astype(int)

        prob_matrix.append(probs)
        pred_matrix.append(preds)

prob_matrix = np.stack(prob_matrix,axis=1)
pred_matrix = np.stack(pred_matrix,axis=1)

# --------------------------------------------------
# METRICS
# --------------------------------------------------

wf1 = f1_score(
    Y_test.flatten(),
    pred_matrix.flatten(),
    average="weighted"
)

acc = accuracy_score(
    Y_test.flatten(),
    pred_matrix.flatten()
)

hl = hamming_loss(
    Y_test,
    pred_matrix
)

if len(np.unique(Y_test.flatten())) > 1:
    auc = roc_auc_score(
        Y_test.flatten(),
        prob_matrix.flatten()
    )
else:
    auc = np.nan

print("\nMultitask results:",species)
print("Weighted F1:",wf1)
print("Accuracy:",acc)
print("Hamming loss:",hl)
print("AUC:",auc)

multitask_species_results.append({
    "species":species,
    "weighted_f1":wf1,
    "accuracy":acc,
    "hamming_loss":hl,
    "auc":auc
})


Multitask results: Pseudomonas_Aeruginosa
Weighted F1: 0.8679300584069995
Accuracy: 0.8754325259515571
Hamming loss: 0.1245674740484429
AUC: 0.8642609640254685


In [90]:
multitask_species_results = []

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*80)
    print("MULTITASK MODEL:",species)
    print("="*80)

    # ---------------------------------
    # balanced dataset
    # ---------------------------------

    X_species, Y_species, ab_list = make_balanced_subset_for_species(
        species,
        alpha=ALPHA,
        seed=SEED
    )

    # ---------------------------------
    # build labels
    # ---------------------------------

    patterns = [tuple(row.astype(int)) for row in Y_species]

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    y_lps = np.array([pattern_to_class[p] for p in patterns])

    y_multi = (Y_species.sum(axis=1) > 1).astype(int)

    # ---------------------------------
    # split
    # ---------------------------------

    idx_train, idx_val, idx_test = safe_split_indices(
        y_lps,
        test_size=0.30,
        random_state=SEED
    )

    X_train = X_species[idx_train]
    X_val = X_species[idx_val]
    X_test = X_species[idx_test]

    Y_train = Y_species[idx_train]
    Y_val = Y_species[idx_val]
    Y_test = Y_species[idx_test]

    y_lps_train = y_lps[idx_train]
    y_lps_val = y_lps[idx_val]

    y_multi_train = y_multi[idx_train]
    y_multi_val = y_multi[idx_val]

    # ---------------------------------
    # dataloaders
    # ---------------------------------

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train,dtype=torch.float32),
            torch.tensor(y_lps_train,dtype=torch.long),
            torch.tensor(Y_train,dtype=torch.float32),
            torch.tensor(y_multi_train,dtype=torch.float32)
        ),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_val,dtype=torch.float32),
            torch.tensor(y_lps_val,dtype=torch.long),
            torch.tensor(Y_val,dtype=torch.float32),
            torch.tensor(y_multi_val,dtype=torch.float32)
        ),
        batch_size=128
    )

    # ---------------------------------
    # model
    # ---------------------------------

    model = MultiTaskMLP(
        n_lps_classes=len(unique_patterns),
        n_antibiotics=len(ab_list)
    ).to(device)

    criterion_lps = nn.CrossEntropyLoss()
    criterion_bin = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

    best_loss = np.inf
    patience = 5
    epochs_no_improve = 0
    best_model = None

    # ---------------------------------
    # training
    # ---------------------------------

    for epoch in range(200):

        model.train()

        for Xb,y_lps_b,Yb,y_multi_b in train_loader:

            Xb = Xb.to(device)
            y_lps_b = y_lps_b.to(device)
            Yb = Yb.to(device)
            y_multi_b = y_multi_b.to(device)

            optimizer.zero_grad()

            lps_logits, ab_logits, multi_logits = model(Xb)

            loss_lps = criterion_lps(lps_logits,y_lps_b)

            loss_ab = 0
            for j in range(len(ab_logits)):
                loss_ab += criterion_bin(ab_logits[j],Yb[:,j])

            loss_multi = criterion_bin(multi_logits,y_multi_b)

            loss = loss_lps + loss_ab + loss_multi

            loss.backward()
            optimizer.step()

        # validation

        model.eval()

        val_loss = 0
        n = 0

        with torch.no_grad():

            for Xb,y_lps_b,Yb,y_multi_b in val_loader:

                Xb = Xb.to(device)
                y_lps_b = y_lps_b.to(device)
                Yb = Yb.to(device)
                y_multi_b = y_multi_b.to(device)

                lps_logits, ab_logits, multi_logits = model(Xb)

                loss_lps = criterion_lps(lps_logits,y_lps_b)

                loss_ab = 0
                for j in range(len(ab_logits)):
                    loss_ab += criterion_bin(ab_logits[j],Yb[:,j])

                loss_multi = criterion_bin(multi_logits,y_multi_b)

                loss = loss_lps + loss_ab + loss_multi

                val_loss += loss.item()*len(Xb)
                n += len(Xb)

        val_loss /= n

        print("Epoch",epoch,"Val loss",val_loss)

        if val_loss < best_loss:

            best_loss = val_loss
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print("Early stopping")
            break

    model.load_state_dict(best_model)

    # ---------------------------------
    # TEST EVALUATION
    # ---------------------------------

    model.eval()

    X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)

    with torch.no_grad():

        _, ab_logits, _ = model(X_test_t)

    prob_matrix = []
    pred_matrix = []

    for j in range(len(ab_logits)):

        probs = torch.sigmoid(ab_logits[j]).cpu().numpy()
        preds = (probs>0.5).astype(int)

        prob_matrix.append(probs)
        pred_matrix.append(preds)

    prob_matrix = np.stack(prob_matrix,axis=1)
    pred_matrix = np.stack(pred_matrix,axis=1)

    # metrics

    wf1 = f1_score(
        Y_test.flatten(),
        pred_matrix.flatten(),
        average="weighted"
    )

    acc = accuracy_score(
        Y_test.flatten(),
        pred_matrix.flatten()
    )

    hl = hamming_loss(
        Y_test,
        pred_matrix
    )

    if len(np.unique(Y_test.flatten())) > 1:
        auc = roc_auc_score(
            Y_test.flatten(),
            prob_matrix.flatten()
        )
    else:
        auc = np.nan

    print("\nRESULTS:",species)
    print("Weighted F1:",wf1)
    print("Accuracy:",acc)
    print("Hamming loss:",hl)
    print("AUC:",auc)

    multitask_species_results.append({
        "species":species,
        "weighted_f1":wf1,
        "accuracy":acc,
        "hamming_loss":hl,
        "auc":auc
    })


MULTITASK MODEL: Staphylococcus_Aureus
Epoch 0 Val loss 3.043678971032492
Epoch 1 Val loss 2.970625627063744
Epoch 2 Val loss 2.7955124190581278
Epoch 3 Val loss 2.5667799017937307
Epoch 4 Val loss 2.467440331508468
Epoch 5 Val loss 2.4207103673418744
Epoch 6 Val loss 2.4035973539736815
Epoch 7 Val loss 2.3063319136679974
Epoch 8 Val loss 2.3620090091068318
Epoch 9 Val loss 2.2922632936781517
Epoch 10 Val loss 2.3322141696761527
Epoch 11 Val loss 2.4719808966550625
Epoch 12 Val loss 2.44049034374918
Epoch 13 Val loss 2.481337705637809
Epoch 14 Val loss 2.6233694823369413
Early stopping

RESULTS: Staphylococcus_Aureus
Weighted F1: 0.8778385058808796
Accuracy: 0.889315419065899
Hamming loss: 0.11068458093410109
AUC: 0.8386611669674315

MULTITASK MODEL: Escherichia_Coli
Epoch 0 Val loss 3.4687192359667147
Epoch 1 Val loss 3.325424630990189
Epoch 2 Val loss 3.02759350015876
Epoch 3 Val loss 2.7495230824759838
Epoch 4 Val loss 2.728339093454768
Epoch 5 Val loss 2.647068002250757
Epoch 6 Va

In [87]:
class LPSMLP(nn.Module):

    def __init__(self,n_classes):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(6000,512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(0.2),

            nn.Linear(512,256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(0.2),

            nn.Linear(256,128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(0.2),

            nn.Linear(128,64),
            nn.GELU(),
            nn.LayerNorm(64),

            nn.Linear(64,n_classes)
        )

    def forward(self,x):

        return self.net(x)

In [88]:
lps_species_results = []

for species,ab_list in species_antibiotics.items():

    print("\n"+"="*80)
    print("LPS BASELINE:",species)
    print("="*80)

    # ----------------------------
    # balanced subset
    # ----------------------------

    X_species,Y_species,ab_list = make_balanced_subset_for_species(
        species,
        alpha=ALPHA,
        seed=SEED
    )

    # ----------------------------
    # build LPS classes
    # ----------------------------

    patterns = [tuple(row.astype(int)) for row in Y_species]

    unique_patterns = sorted(list(set(patterns)))

    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    class_to_pattern = {i:p for p,i in pattern_to_class.items()}

    y_lps = np.array([pattern_to_class[p] for p in patterns])

    print("Samples:",len(X_species))
    print("LPS classes:",len(unique_patterns))

    # ----------------------------
    # split
    # ----------------------------

    idx_train,idx_val,idx_test = safe_split_indices(
        y_lps,
        test_size=0.30,
        random_state=SEED
    )

    X_train = X_species[idx_train]
    X_val = X_species[idx_val]
    X_test = X_species[idx_test]

    y_train = y_lps[idx_train]
    y_val = y_lps[idx_val]
    y_test = y_lps[idx_test]

    Y_test = Y_species[idx_test]

    # ----------------------------
    # dataloaders
    # ----------------------------

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train,dtype=torch.float32),
            torch.tensor(y_train,dtype=torch.long)
        ),
        batch_size=128,
        shuffle=True
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_val,dtype=torch.float32),
            torch.tensor(y_val,dtype=torch.long)
        ),
        batch_size=128
    )

    # ----------------------------
    # model
    # ----------------------------

    model = LPSMLP(len(unique_patterns)).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4
    )

    best_loss = np.inf
    patience = 5
    epochs_no_improve = 0
    best_model = None

    # ----------------------------
    # training
    # ----------------------------

    for epoch in range(200):

        model.train()

        for Xb,yb in train_loader:

            Xb = Xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(Xb)

            loss = criterion(logits,yb)

            loss.backward()

            optimizer.step()

        # validation

        model.eval()

        val_loss = 0
        n = 0

        with torch.no_grad():

            for Xb,yb in val_loader:

                Xb = Xb.to(device)
                yb = yb.to(device)

                logits = model(Xb)

                loss = criterion(logits,yb)

                val_loss += loss.item()*len(Xb)
                n += len(Xb)

        val_loss /= n

        print("Epoch",epoch,"Val loss",val_loss)

        if val_loss < best_loss:

            best_loss = val_loss
            epochs_no_improve = 0
            best_model = copy.deepcopy(model.state_dict())

        else:

            epochs_no_improve += 1

        if epochs_no_improve >= patience:

            print("Early stopping")
            break

    model.load_state_dict(best_model)

    # ----------------------------
    # TEST PREDICTION
    # ----------------------------

    model.eval()

    X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)

    with torch.no_grad():

        logits = model(X_test_t)

        probs = torch.softmax(logits,dim=1).cpu().numpy()

        preds = np.argmax(probs,axis=1)

    pred_patterns = np.array([
        class_to_pattern[c] for c in preds
    ])

    patterns_matrix = np.array([
        class_to_pattern[i] for i in range(len(unique_patterns))
    ])

    prob_matrix = probs @ patterns_matrix

    # ----------------------------
    # metrics
    # ----------------------------

    wf1 = f1_score(
        Y_test.flatten(),
        pred_patterns.flatten(),
        average="weighted"
    )

    acc = accuracy_score(
        Y_test.flatten(),
        pred_patterns.flatten()
    )

    hl = hamming_loss(
        Y_test,
        pred_patterns
    )

    if len(np.unique(Y_test.flatten())) > 1:

        auc = roc_auc_score(
            Y_test.flatten(),
            prob_matrix.flatten()
        )

    else:

        auc = np.nan

    print("\nRESULTS:",species)
    print("Weighted F1:",wf1)
    print("Accuracy:",acc)
    print("Hamming loss:",hl)
    print("AUC:",auc)

    lps_species_results.append({
        "species":species,
        "weighted_f1":wf1,
        "accuracy":acc,
        "hamming_loss":hl,
        "auc":auc
    })


LPS BASELINE: Staphylococcus_Aureus
Samples: 3472
LPS classes: 8
Epoch 0 Val loss 1.3256569080297869
Epoch 1 Val loss 1.2723316514789487
Epoch 2 Val loss 1.1875471739302212
Epoch 3 Val loss 1.131942701431245
Epoch 4 Val loss 1.0816556258759893
Epoch 5 Val loss 1.0765891107167485
Epoch 6 Val loss 1.0843886496238195
Epoch 7 Val loss 1.031797950647614
Epoch 8 Val loss 1.0756989522996196
Epoch 9 Val loss 1.1259174182181624
Epoch 10 Val loss 1.095157095887153
Epoch 11 Val loss 1.139320049770963
Epoch 12 Val loss 1.1620512914749117
Early stopping

RESULTS: Staphylococcus_Aureus
Weighted F1: 0.868443313992258
Accuracy: 0.8841970569417786
Hamming loss: 0.11580294305822136
AUC: 0.8275844289765404

LPS BASELINE: Escherichia_Coli
Samples: 4750
LPS classes: 8
Epoch 0 Val loss 1.2372085632902852
Epoch 1 Val loss 1.1924910424800401
Epoch 2 Val loss 1.1240479597884618
Epoch 3 Val loss 1.0430218255921695
Epoch 4 Val loss 1.0600123177753407
Epoch 5 Val loss 1.0021459279435405
Epoch 6 Val loss 1.021704

In [91]:
from sklearn.model_selection import KFold

multitask_species_results = []

# search space for loss weights
weight_grid = [
    (1.0,1.0,0.5),
    (1.0,1.0,1.0),
    (1.0,0.5,0.5),
    (1.0,0.5,1.0),
    (0.5,1.0,0.5),
]

for species, ab_list in species_antibiotics.items():

    print("\n"+"="*80)
    print("MULTITASK MODEL:",species)
    print("="*80)

    X_species, Y_species, ab_list = make_balanced_subset_for_species(
        species,
        alpha=ALPHA,
        seed=SEED
    )

    patterns = [tuple(row.astype(int)) for row in Y_species]
    unique_patterns = sorted(list(set(patterns)))
    pattern_to_class = {p:i for i,p in enumerate(unique_patterns)}

    y_lps = np.array([pattern_to_class[p] for p in patterns])
    y_multi = (Y_species.sum(axis=1) > 1).astype(int)

    # ---------------------------------
    # train / test split
    # ---------------------------------

    idx_train, idx_val, idx_test = safe_split_indices(
        y_lps,
        test_size=0.30,
        random_state=SEED
    )

    X_train = X_species[idx_train]
    X_test = X_species[idx_test]

    Y_train = Y_species[idx_train]
    Y_test = Y_species[idx_test]

    y_lps_train = y_lps[idx_train]
    y_multi_train = y_multi[idx_train]

    # ---------------------------------
    # CROSS VALIDATION
    # ---------------------------------

    print("\nRunning 5-fold CV for loss weights")

    best_weights = None
    best_score = -np.inf

    kf = KFold(n_splits=5,shuffle=True,random_state=SEED)

    for w_lps, w_ab, w_multi in weight_grid:

        cv_scores = []

        for train_idx, val_idx in kf.split(X_train):

            X_tr = X_train[train_idx]
            X_val = X_train[val_idx]

            Y_tr = Y_train[train_idx]
            Y_val = Y_train[val_idx]

            y_lps_tr = y_lps_train[train_idx]
            y_lps_val = y_lps_train[val_idx]

            y_multi_tr = y_multi_train[train_idx]
            y_multi_val = y_multi_train[val_idx]

            train_loader = DataLoader(
                TensorDataset(
                    torch.tensor(X_tr,dtype=torch.float32),
                    torch.tensor(y_lps_tr,dtype=torch.long),
                    torch.tensor(Y_tr,dtype=torch.float32),
                    torch.tensor(y_multi_tr,dtype=torch.float32)
                ),
                batch_size=128,
                shuffle=True
            )

            val_loader = DataLoader(
                TensorDataset(
                    torch.tensor(X_val,dtype=torch.float32),
                    torch.tensor(y_lps_val,dtype=torch.long),
                    torch.tensor(Y_val,dtype=torch.float32),
                    torch.tensor(y_multi_val,dtype=torch.float32)
                ),
                batch_size=128
            )

            model = MultiTaskMLP(
                n_lps_classes=len(unique_patterns),
                n_antibiotics=len(ab_list)
            ).to(device)

            optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

            criterion_lps = nn.CrossEntropyLoss()
            criterion_bin = nn.BCEWithLogitsLoss()

            for epoch in range(50):

                model.train()

                for Xb,y_lps_b,Yb,y_multi_b in train_loader:

                    Xb = Xb.to(device)
                    y_lps_b = y_lps_b.to(device)
                    Yb = Yb.to(device)
                    y_multi_b = y_multi_b.to(device)

                    optimizer.zero_grad()

                    lps_logits, ab_logits, multi_logits = model(Xb)

                    loss_lps = criterion_lps(lps_logits,y_lps_b)

                    loss_ab = 0
                    for j in range(len(ab_logits)):
                        loss_ab += criterion_bin(ab_logits[j],Yb[:,j])

                    loss_multi = criterion_bin(multi_logits,y_multi_b)

                    loss = (
                        w_lps*loss_lps +
                        w_ab*loss_ab +
                        w_multi*loss_multi
                    )

                    loss.backward()
                    optimizer.step()

            # validation prediction

            model.eval()

            X_val_t = torch.tensor(X_val,dtype=torch.float32).to(device)

            with torch.no_grad():

                _, ab_logits, _ = model(X_val_t)

            preds = []

            for j in range(len(ab_logits)):

                p = torch.sigmoid(ab_logits[j]).cpu().numpy()
                preds.append((p>0.5).astype(int))

            preds = np.stack(preds,axis=1)

            score = f1_score(
                Y_val.flatten(),
                preds.flatten(),
                average="weighted"
            )

            cv_scores.append(score)

        mean_score = np.mean(cv_scores)

        print("weights", (w_lps,w_ab,w_multi), "WF1", mean_score)

        if mean_score > best_score:

            best_score = mean_score
            best_weights = (w_lps,w_ab,w_multi)

    print("Best weights:",best_weights)

    w_lps,w_ab,w_multi = best_weights

    # ---------------------------------
    # FINAL TRAINING
    # ---------------------------------

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_train,dtype=torch.float32),
            torch.tensor(y_lps_train,dtype=torch.long),
            torch.tensor(Y_train,dtype=torch.float32),
            torch.tensor(y_multi_train,dtype=torch.float32)
        ),
        batch_size=128,
        shuffle=True
    )

    model = MultiTaskMLP(
        n_lps_classes=len(unique_patterns),
        n_antibiotics=len(ab_list)
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)

    criterion_lps = nn.CrossEntropyLoss()
    criterion_bin = nn.BCEWithLogitsLoss()

    for epoch in range(100):

        model.train()

        for Xb,y_lps_b,Yb,y_multi_b in train_loader:

            Xb = Xb.to(device)
            y_lps_b = y_lps_b.to(device)
            Yb = Yb.to(device)
            y_multi_b = y_multi_b.to(device)

            optimizer.zero_grad()

            lps_logits, ab_logits, multi_logits = model(Xb)

            loss_lps = criterion_lps(lps_logits,y_lps_b)

            loss_ab = 0
            for j in range(len(ab_logits)):
                loss_ab += criterion_bin(ab_logits[j],Yb[:,j])

            loss_multi = criterion_bin(multi_logits,y_multi_b)

            loss = (
                w_lps*loss_lps +
                w_ab*loss_ab +
                w_multi*loss_multi
            )

            loss.backward()
            optimizer.step()

    # ---------------------------------
    # TEST EVALUATION
    # ---------------------------------

    model.eval()

    X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)

    with torch.no_grad():

        _, ab_logits, _ = model(X_test_t)

    prob_matrix = []
    pred_matrix = []

    for j in range(len(ab_logits)):

        probs = torch.sigmoid(ab_logits[j]).cpu().numpy()
        preds = (probs>0.5).astype(int)

        prob_matrix.append(probs)
        pred_matrix.append(preds)

    prob_matrix = np.stack(prob_matrix,axis=1)
    pred_matrix = np.stack(pred_matrix,axis=1)

    wf1 = f1_score(Y_test.flatten(),pred_matrix.flatten(),average="weighted")
    acc = accuracy_score(Y_test.flatten(),pred_matrix.flatten())
    hl = hamming_loss(Y_test,pred_matrix)

    if len(np.unique(Y_test.flatten())) > 1:
        auc = roc_auc_score(Y_test.flatten(),prob_matrix.flatten())
    else:
        auc = np.nan

    print("\nRESULTS:",species)
    print("Weighted F1:",wf1)
    print("Accuracy:",acc)
    print("Hamming loss:",hl)
    print("AUC:",auc)

    multitask_species_results.append({
        "species":species,
        "weighted_f1":wf1,
        "accuracy":acc,
        "hamming_loss":hl,
        "auc":auc,
        "best_weights":best_weights
    })


MULTITASK MODEL: Staphylococcus_Aureus

Running 5-fold CV for loss weights
weights (1.0, 1.0, 0.5) WF1 0.8548692888126194
weights (1.0, 1.0, 1.0) WF1 0.8482590508771164
weights (1.0, 0.5, 0.5) WF1 0.8540045384241587
weights (1.0, 0.5, 1.0) WF1 0.8574877288993367
weights (0.5, 1.0, 0.5) WF1 0.8515355492369563
Best weights: (1.0, 0.5, 1.0)

RESULTS: Staphylococcus_Aureus
Weighted F1: 0.8628273895274753
Accuracy: 0.8701215611004478
Hamming loss: 0.12987843889955214
AUC: 0.802526424336169

MULTITASK MODEL: Escherichia_Coli

Running 5-fold CV for loss weights
weights (1.0, 1.0, 0.5) WF1 0.7955379317027897
weights (1.0, 1.0, 1.0) WF1 0.8024568203497623
weights (1.0, 0.5, 0.5) WF1 0.802113489291511
weights (1.0, 0.5, 1.0) WF1 0.8019697562909162
weights (0.5, 1.0, 0.5) WF1 0.8023303378263243
Best weights: (1.0, 1.0, 1.0)

RESULTS: Escherichia_Coli
Weighted F1: 0.8037643222484487
Accuracy: 0.8083216456287985
Hamming loss: 0.1916783543712015
AUC: 0.8135435542507169

MULTITASK MODEL: Klebsiella_